<a href="https://colab.research.google.com/github/aymanr8-dot/DS-2002-Walmart-Hurricane-Project_Data-Demons-/blob/main/notebooks/team_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Walmart Hurricane Analytics — DS 2002 Midterm
**Team Data Demons** · Esther Man & Ayman Haque · Fall 2026

**What this notebook does.** Cleans five messy Walmart data files, joins them to real weather from the Open-Meteo API,
answers the five project questions, and ends with what we tell leadership.

**Sources.** The five project data files (read from the team repo), the Open-Meteo API, the National Hurricane Center's official best track for Hurricane Helene (HURDAT2), and the *Walmart Case*. The transactions cover
**September 1–30, 2024** across 12 Florida stores. Landfall is **September 26, 2024** (given in the project brief); NHC places it at 11:10 pm EDT near Perry, in Florida's
Big Bend. Q1's surge window is the brief's **7 days before landfall (Sept 19–25)**. Q4 measures every store's distance to
the official NHC track. The Open-Meteo weather data cross-checks both.

| Section | What it does |
|---|---|
| 0 | Setup |
| 1 | Data loading and cleaning pipeline (functions), row counts before/after |
| 2 | Profile raw vs. clean, five worst problems, SKU mapping table, decision log, daily volume chart |
| 3 | Weather API (retries, `source` column, GMT → store-local time, checked join, storm timing) |
| 4 | **Q1** Demand surge |
| 5 | **Q2** SKU consolidation |
| 6 | **Q3** Weather correlation and lag |
| 7 | **Q4** Stores and geography (distance to the official NHC track), plus distribution-center capacity |
| 8 | **Q5** Signal or artifact: the board-game spike (schemas first, then SQL on the SQLite database) |
| 9 | What we tell leadership |
| 10 | Full decision log (File · Issue · Decision · Rows affected · Reason) and the final self-check |
| 11 | Reflection |


## 0. Setup

In [ ]:
import os, re, time, json, sqlite3, urllib.parse, urllib.request
from datetime import date

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mtick
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# ---- Analysis constants (every one of these is a decision; see the decision log) ----
LANDFALL = pd.Timestamp("2024-09-26")             # Hurricane Helene landfall date, given in the project brief
SURGE_DAYS = 7                                     # the brief's surge period: the 7 days before landfall
SURGE_START = LANDFALL - pd.Timedelta(days=SURGE_DAYS)   # Sept 19
SURGE_END = LANDFALL - pd.Timedelta(days=1)              # Sept 25 (landfall day itself is not in the window)
BASELINE_START, BASELINE_END = pd.Timestamp("2024-09-01"), pd.Timestamp("2024-09-14")   # our choice, defended in Q1
SURGE_SEARCH_START = pd.Timestamp("2024-09-15")  # first day after the baseline (used to date when demand broke out)
DATA_DIR = "data"                                # local folder for the weather cache
os.makedirs(DATA_DIR, exist_ok=True)

# ---- Chart style: one validated categorical order, thin marks, recessive grid ----
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MUTED = "#8a8984"
plt.rcParams.update({
    "figure.figsize": (10, 4.8), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 10.5, "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "legend.frameon": False, "lines.linewidth": 2, "axes.axisbelow": True,
})

def shade_windows(ax, arrival=None, alpha=0.10):
    """Shade the baseline window; if given, mark landfall day."""
    ax.axvspan(BASELINE_START, BASELINE_END + pd.Timedelta(days=1), color=MUTED, alpha=alpha, lw=0)
    ymax = ax.get_ylim()[1]
    ax.text(BASELINE_START + pd.Timedelta(hours=6), ymax * 0.97, "baseline", va="top", fontsize=9, color="#52514e")
    if arrival is not None:
        ax.axvline(arrival + pd.Timedelta(hours=12), color="#52514e", ls="--", lw=1)
        ax.text(arrival + pd.Timedelta(hours=18), ymax * 0.97, "landfall", va="top", ha="left", fontsize=9, color="#52514e")

## 1. Data loading and cleaning pipeline
**Where each source is loaded:** `walmart_transactions.csv` and `store_locations.csv` here; `product_catalog.csv` in 2.1; the Open-Meteo weather API in section 3; `distribution_centers.csv` in 7.3; `inventory_and_sales.db` (queried with `sqlite3`) in section 8.

One function per problem, chained in `clean_transactions()`. Every function writes one row to the shared
`decision_log`, which covers every file we touch: **File · Issue · Decision · Rows affected · Reason** (plus rows after). Nothing is silently dropped except exact duplicates: returns and rows with no timestamp are **kept and flagged**.

In [ ]:
DATA_URL = ('https://raw.githubusercontent.com/aymanr8-dot/DS-2002-Walmart-Hurricane-Project_Data-Demons-/'
            'main/data/')

REPO_NAMES = {'distribution_centers.csv': 'distribution_centers (1).csv',
              'inventory_and_sales.db': 'inventory_and_sales (1).db'}

def data_path(name, folders=('data', '../data', '/content')):
    """Local copy if there is one (repo checkout, or a file uploaded to Colab), otherwise the team repo on GitHub."""
    for candidate in (name, REPO_NAMES.get(name)):
        if candidate is None:
            continue
        for folder in folders:
            p = os.path.join(folder, candidate)
            if os.path.exists(p):
                return p
    return DATA_URL + urllib.parse.quote(REPO_NAMES.get(name, name))

raw = pd.read_csv(data_path('walmart_transactions.csv'), dtype=str)
stores_raw = pd.read_csv(data_path('store_locations.csv'), dtype=str)

decision_log = []

def log_decision(file, issue, decision, rows_affected, reason, rows_after=None):
    """One shared log for every file: File | Issue | Decision | Rows affected | Reason (+ rows after)."""
    decision_log.append({'file': file, 'issue': issue, 'decision': decision,
                         'rows_affected': int(rows_affected), 'reason': reason,
                         'rows_after': None if rows_after is None else int(rows_after)})

TX = 'walmart_transactions.csv'

def drop_duplicates(df):
    dupes = df.duplicated()
    out = df[~dupes].copy()
    log_decision(TX, 'Duplicate transactions', 'Dropped exact duplicate rows (same transaction_id and every field)',
                 dupes.sum(), 'An identical row is the same sale recorded twice; keeping it double-counts units', len(out))
    return out

def clean_store_id(s):
    """' FL-330', 'FL_330', 'FL330', 'fl-330' -> 'FL-330'"""
    s = s.astype(str).str.strip().str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
    return s.str.replace(r'^([A-Z]{2})(\d+)$', r'\1-\2', regex=True)

def standardize_store_ids(df):
    before = df['store_id']
    df['store_id'] = clean_store_id(before)
    log_decision(TX, 'Inconsistent store IDs', f"Stripped spaces, upper-cased, rewrote to FL-### "
                 f"({before.nunique()} spellings -> {df['store_id'].nunique()} stores)",
                 (before != df['store_id']).sum(),
                 'Same rule as store_locations.csv, and matches the SQLite tables (already FL-###), so every join lines up',
                 len(df))
    return df

def parse_unit_price(df):
    had_dollar = df['unit_price'].str.contains(r'\$', na=False)
    df['unit_price'] = pd.to_numeric(df['unit_price'].str.replace(r'[$,\s]', '', regex=True),
                                     errors='coerce')
    assert df['unit_price'].notna().all(), 'some unit_price values did not convert'
    log_decision(TX, 'Prices stored as strings', "Removed leading '$' and converted text to float",
                 had_dollar.sum(), 'Text prices cannot be multiplied into revenue or compared; every value converted (asserted)',
                 len(df))
    return df

def flag_returns(df):
    df['quantity'] = pd.to_numeric(df['quantity']).astype(int)
    df['is_return'] = df['quantity'] < 0
    log_decision(TX, 'Negative quantities', 'Treated as returns: kept, flagged is_return=True, excluded from demand counts',
                 df['is_return'].sum(), 'A return is not demand; dropping it would hide refunds, netting it would shrink the surge',
                 len(df))
    return df

TS_FORMATS = ['%Y-%m-%d %H:%M:%S',   # 2024-09-13 20:30:20
              '%Y-%m-%dT%H:%M:%S',   # 2024-09-24T09:48:13
              '%m/%d/%Y %H:%M',      # 09/26/2024 09:26
              '%m-%d-%Y %H:%M:%S',   # 09-23-2024 18:24:47
              '%d-%b-%Y %H:%M']      # 21-Sep-2024 19:46

def parse_timestamps(df):
    s = df['timestamp'].str.strip()
    parsed = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
    for fmt in TS_FORMATS:
        parsed = parsed.fillna(pd.to_datetime(s, format=fmt, errors='coerce'))
    unparsed = s.notna() & parsed.isna()
    assert unparsed.sum() == 0, f'{unparsed.sum()} timestamps in an unknown format'
    reformatted = s.notna() & ~s.str.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$', na=False)
    df['timestamp'] = parsed
    log_decision(TX, 'Inconsistent date formats', f'Parsed {len(TS_FORMATS)} explicit text formats to datetime (store-local time)',
                 reformatted.sum(), 'Every by-day result needs one type; explicit formats mean day and month are never guessed',
                 len(df))
    df['timestamp_missing'] = df['timestamp'].isna()
    log_decision(TX, 'Missing timestamps', 'Flagged timestamp_missing=True, not dropped or imputed: count toward product '
                 'totals, left out of anything by date', df['timestamp_missing'].sum(),
                 'The sale happened, but its day is unknown; imputing a day could move it across the surge window', len(df))
    return df

CATEGORY_MAP = {'emergency': 'Emergency', 'emergency supplies': 'Emergency',
                'grocery': 'Grocery', 'groceries': 'Grocery',
                'electronics': 'Electronics',
                'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
                'household': 'Household', 'household essentials': 'Household',
                'baby': 'Baby', 'baby products': 'Baby',
                'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast',
                'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
                'snacks': 'Snacks',
                'health': 'Health', 'health & wellness': 'Health'}

def standardize_categories(df):
    before = df['category']
    df['category'] = before.str.strip().str.lower().map(CATEGORY_MAP)
    assert df['category'].notna().all(), 'unmapped category label'
    log_decision(TX, 'Inconsistent categories', f"Mapped {before.nunique()} category spellings to {df['category'].nunique()} categories",
                 (before != df['category']).sum(), 'Case and synonym variants split one category; Q1 ranks categories, so one label each',
                 len(df))
    df['product_name'] = df['product_name'].str.strip()
    return df

STORE_TZ = {'FL-710': 'America/Chicago'}   # Pensacola is on Central time; every other store is Eastern
GEOCODE_URL = 'https://geocoding-api.open-meteo.com/v1/search'   # same provider as the weather data, no key

def geocode_city(city, state='Florida'):
    """City -> (lat, lon) from Open-Meteo's geocoding API, keeping only matches in the right US state."""
    r = requests.get(GEOCODE_URL, params={'name': city, 'count': 20}, timeout=30)
    r.raise_for_status()
    hits = [h for h in r.json().get('results', []) if h.get('admin1') == state and h.get('country_code') == 'US']
    assert hits, f'no geocoding match for {city}, {state}'
    return hits[0]['latitude'], hits[0]['longitude']

def clean_stores(stores):
    s = stores.copy()
    s['store_id'] = clean_store_id(s['store_id'])
    s = s.drop_duplicates('store_id').copy()            # FL239 / FL_287 were repeat rows
    s['state'] = 'FL'                                    # 'Florida', 'Fla.' -> 'FL'
    s['region'] = s['region'].str.strip()
    s[['latitude', 'longitude']] = s[['latitude', 'longitude']].astype(float)
    for i in s.index[s['latitude'].isna()]:              # Tallahassee had blank coordinates: look them up
        s.loc[i, ['latitude', 'longitude']] = geocode_city(s.loc[i, 'city'])
    s['tz'] = s['store_id'].map(STORE_TZ).fillna('America/New_York')
    return s.reset_index(drop=True)

def clean_transactions(df):
    df = drop_duplicates(df)
    df = standardize_store_ids(df)
    df = parse_unit_price(df)
    df = flag_returns(df)
    df = parse_timestamps(df)
    df = standardize_categories(df)
    return df.reset_index(drop=True)

sales = clean_transactions(raw)
stores = clean_stores(stores_raw)

print(f'rows before cleaning: {len(raw):,}')
print(f'rows after cleaning:  {len(sales):,}')
print(f'rows removed:         {len(raw) - len(sales):,}')
print(f"returns flagged (kept): {sales['is_return'].sum():,}")
print(f"missing timestamps flagged (kept): {sales['timestamp_missing'].sum():,}")
print(f"store file: {len(stores_raw)} rows -> {len(stores)} stores after dedupe")
unjoined = set(sales['store_id']) - set(stores['store_id'])
print('store_ids that do not join to the store file:', unjoined or 'none')
print('date range:', sales['timestamp'].min(), 'to', sales['timestamp'].max())
sales.head()

## 2. Profile: raw vs. clean
The cleaning functions copy the data, so `raw` is still the untouched file. Profiling both side by side shows each fix landed.

In [ ]:
NULL_TOKENS = {"", "NULL", "N/A", "NA", "NaN", "nan", "None", "null"}

def profile(df, name="df"):
    """Rows, columns, types, nulls (counting text null tokens), distinct values, and a few examples per column."""
    out = []
    for c in df.columns:
        s = df[c]
        is_null = s.isna() | s.astype(str).str.strip().isin(NULL_TOKENS)
        out.append({
            "column": c,
            "dtype": str(s.dtype),
            "null_pct": round(100 * is_null.mean(), 2),
            "distinct": s[~is_null].nunique(),
            "examples": list(s[~is_null].drop_duplicates().head(4)),
        })
    print(f"{name}: {len(df):,} rows x {df.shape[1]} cols | exact duplicate rows: {df.duplicated().sum()}")
    return pd.DataFrame(out).set_index("column")

display(profile(raw, "walmart_transactions (raw)"))
display(profile(sales, "sales (clean)"))

In [ ]:
ts_shapes = raw["timestamp"].str.replace(r"\d", "9", regex=True).value_counts(dropna=False)
print("Raw timestamp shapes (digits replaced by 9; NaN = empty/NULL/N/A/NaN/nan):\n", ts_shapes, "\n")
print("store_id spellings:", raw["store_id"].nunique(), "->", sales["store_id"].nunique())
print("category spellings:", raw["category"].nunique(), "->", sales["category"].nunique())
print("product_name spellings:", raw["product_name"].nunique(), "| SKUs:", raw["sku"].nunique())
print("unit_price with $ sign:", raw["unit_price"].str.contains(r"\$").sum())
display(profile(stores_raw, "store_locations (raw)"))

### The five worst problems (from the profile)
1. **Timestamps come in five formats plus five spellings of "missing".** `2024-09-12 08:15:00`, `2024-09-12T08:15:00`, `09/12/2024 08:15`, `12-Sep-2024 08:15`, `09-12-2024 08:15:00`, and 886 raw rows (3.5%; 884 after dropping duplicates, 881 of them sales rather than returns) that are empty, `NULL`, `N/A`, `NaN` or `nan`. Q1 and the weather join both run on dates, so this comes first.
2. **The same product sits under several SKUs.** 45 SKUs, but only 20 real products. Strawberry Pop-Tarts alone appears as `PT-12`, `SKU#459812`, `POPTART-STRAW` and `PT_STRAWBERRY`. The product name is spelled 63 ways.
3. **`store_id` has 60 spellings for 12 stores.** `FL-239`, `fl-239`, `FL_239`, `FL239`, and `" FL-239"` with a leading space. The store file itself has two duplicate stores (`FL239`, `FL_287`), and Tallahassee has no coordinates.
4. **`category` has 37 spellings for 10 categories.** Case (`EMERGENCY` / `emergency`) and synonyms (`Emergency Supplies`, `Groceries`, `Games & Entertainment`, `Alcoholic Beverages`, `Baby Products`).
5. **Numbers stored as text.** `unit_price` is text, and 3,029 values (12%) start with `$` in the raw file (3,026 once the 30 duplicates are dropped, which is the count in the decision log). `quantity` has 130 negative values (returns), and 30 rows are exact duplicates.

### 2.1 SKU consolidation mapping (needed by Q2)
**Method.** In the transaction file, each SKU is sold under several product-name spellings, and those spellings are *shared*
across SKUs (`HD Tarp` appears under both `TARP-HD` and `HDTARP-01`). We link SKUs that share a normalized product name
(a union-find over the SKU–name graph). Each connected group is one real product. We do not have to
guess that `PT-12` and `SKU#459812` are the same Pop-Tart: the cashiers' name strings tell us. Every group is then reviewed and given a canonical name.

**Mapping table, not a dictionary.** `SKU_MAP` is a table with one row per SKU (sku → product_id, product, category),
built from the data. The only hand-entered part is the readable name for each group (`CANONICAL`); the grouping itself
comes from the POS records. In 2.2 a **left join** lists any SKU the table doesn't cover (none), and we assert that the
join keeps every row and every unit, so consolidation cannot create sales.


In [ ]:
def norm_text(s):
    return s.astype(str).str.strip().str.lower().str.replace(r"\s+", " ", regex=True)

def build_sku_map(tx, catalog):
    """Group SKUs that share a normalized product name in POS data; return one row per SKU."""
    pairs = pd.DataFrame({"sku": tx["sku"].str.strip().str.upper(), "name": norm_text(tx["product_name"])}).drop_duplicates()
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for sku, name in pairs.itertuples(index=False):
        a, b = find("S:" + sku), find("N:" + name)
        if a != b:
            parent[a] = b
    skus = sorted(set(pairs["sku"]) | set(catalog["sku"].str.strip().str.upper()))
    groups = {}
    for sku in skus:
        groups.setdefault(find("S:" + sku), []).append(sku)
    return groups

# Hand-reviewed canonical names, keyed by any one SKU in the group
CANONICAL = {
    "PT-12": ("P01", "Pop-Tarts Strawberry"),        "WTR-24": ("P02", "Bottled Water 24-Pack"),
    "BATT-100": ("P03", "AA Batteries 4-Pack"),      "FLASH-01": ("P04", "LED Flashlight"),
    "BEER-24": ("P05", "Light Beer 24-Pack"),        "BG-MONO": ("P06", "Monopoly Board Game"),
    "CARDS-01": ("P07", "Playing Cards"),            "STREAM-01": ("P08", "Roku Streaming Stick"),
    "SOUP-CAN": ("P09", "Canned Soup"),              "BREAD-WH": ("P10", "White Bread"),
    "PB-16": ("P11", "Peanut Butter 16oz"),          "CNDL-EM": ("P12", "Emergency Candles"),
    "FAID-01": ("P13", "First Aid Kit"),             "CHRG-PORT": ("P14", "Portable Phone Charger"),
    "CHIPS-VP": ("P15", "Chips Variety Pack"),       "PTOWEL-6": ("P16", "Paper Towels 6-Roll"),
    "TP-12": ("P17", "Toilet Paper 12-Roll"),        "DIAP-S3": ("P18", "Diapers Size 3"),
    "BFORM-01": ("P19", "Infant Formula"),           "TARP-HD": ("P20", "Heavy Duty Tarp"),
}

def sku_mapping_table(tx_raw, cat_raw):
    groups = build_sku_map(tx_raw, cat_raw)
    rows = []
    for members in groups.values():
        keys = [s for s in members if s in CANONICAL]
        assert len(keys) == 1, f"group {members} needs exactly one canonical entry, found {keys}"
        pid, pname = CANONICAL[keys[0]]
        rows += [{"sku": s, "product_id": pid, "product": pname, "skus_in_group": len(members)} for s in members]
    m = pd.DataFrame(rows).sort_values(["product_id", "sku"]).reset_index(drop=True)
    # Category per product: normalize every category label seen for its SKUs; it must be unanimous.
    t = pd.DataFrame({"sku": tx_raw["sku"].str.strip().str.upper(), "category": norm_text(tx_raw["category"]).map(CATEGORY_MAP)})
    cat = t.merge(m, on="sku").groupby("product_id")["category"].agg(lambda x: sorted(x.dropna().unique()))
    assert (cat.str.len() == 1).all(), f"products with conflicting categories:\n{cat[cat.str.len() != 1]}"
    m["category"] = m["product_id"].map(cat.str[0])
    return m

catalog_raw = pd.read_csv(data_path('product_catalog.csv'), dtype=str)
SKU_MAP = sku_mapping_table(raw, catalog_raw)
print(f"{SKU_MAP['sku'].nunique()} SKUs -> {SKU_MAP['product_id'].nunique()} products")
SKU_MAP

### 2.2 Apply the mapping and finish the analysis table
Adds the canonical product to every row, a calendar `date`, and `revenue`. Also logs the store-file fixes made inside
`clean_stores()`, so the decision log covers every file we touched.

In [ ]:
n_before, units_before, revenue_before = len(sales), sales["quantity"].sum(), (sales["quantity"] * sales["unit_price"]).sum()

# Left join (as in the 10/26 clinic): which SKUs in the sales data have no row in the mapping table?
sku_check = sales[["sku"]].drop_duplicates().merge(SKU_MAP[["sku"]], on="sku", how="left", indicator=True)
unmapped = sku_check.loc[sku_check["_merge"] == "left_only", "sku"].tolist()
print("SKUs in the sales data with no row in SKU_MAP (left join):", unmapped or "none")
assert not unmapped, f"unmapped SKUs: {unmapped}"

sales = sales.merge(SKU_MAP[["sku", "product_id", "product", "category"]].rename(columns={"category": "product_category"}),
                    on="sku", how="left", validate="many_to_one")
assert len(sales) == n_before and sales["product_id"].notna().all(), "SKU mapping changed the row count or left a SKU unmapped"
assert (sales["category"] == sales["product_category"]).all(), "row category disagrees with the product's category"
# Consolidation must not create (or lose) sales: same rows, same units, same revenue
assert sales["quantity"].sum() == units_before, "SKU mapping changed total units"
assert np.isclose((sales["quantity"] * sales["unit_price"]).sum(), revenue_before), "SKU mapping changed total revenue"
print(f"Units before / after the SKU merge: {units_before:,} / {sales['quantity'].sum():,} (identical)")
sales = sales.drop(columns="product_category")
sales["product_name_raw"] = sales["product_name"]
sales["product_name"] = sales["product"]
sales = sales.drop(columns="product")
log_decision(TX, "Fragmented SKUs", f"Mapped {sales['sku'].nunique()} SKUs to {sales['product_id'].nunique()} products via the "
             "SKU_MAP table (left join, many_to_one); canonical name in product_name, original kept in product_name_raw",
             (sales["product_name_raw"] != sales["product_name"]).sum(),
             "One product under 2-4 SKUs hides its real volume (Q2); groups come from shared cashier product names, "
             "hand-reviewed; units and revenue asserted unchanged", len(sales))

sales["date"] = sales["timestamp"].dt.normalize()
sales["revenue"] = sales["quantity"] * sales["unit_price"]

# Store-file decisions made inside clean_stores()
ST = "store_locations.csv"
log_decision(ST, "Inconsistent store IDs / duplicate stores", "Same clean_store_id() rule as the transactions; dropped repeat "
             "rows (FL239 = FL-239, FL_287 = FL-287)", len(stores_raw) - len(stores),
             "One row per store so the reference join is many_to_one and IDs match the transactions exactly", len(stores))
log_decision(ST, "Inconsistent state values", "Set 'Florida' / 'Fla.' to 'FL'", (stores_raw["state"] != "FL").sum(),
             "Every store is in Florida; one spelling", len(stores))
log_decision(ST, "Missing values (coordinates)", "FL-619 Tallahassee had blank lat/lon: looked up with Open-Meteo's geocoding API",
             stores_raw["latitude"].isna().sum(), "Q3 weather calls and Q4 distances need coordinates for every store; "
             "same provider as the weather data", len(stores))
log_decision(ST, "Time zone not recorded", "Added each store's time zone: Pensacola FL-710 is Central, the rest Eastern",
             len(stores), "Sales are store-local and the weather API returns GMT; days must be aligned per store (Q3)", len(stores))

# Analysis views used by Q1-Q2
demand = sales[~sales["is_return"]]                      # units sold (returns excluded)
dated_sales = demand[~demand["timestamp_missing"]]       # ... that also have a usable date

# Catalog with the canonical product attached (Q2 uses it to show the catalog's internal conflicts)
catalog = catalog_raw.assign(sku=catalog_raw["sku"].str.strip().str.upper()).merge(
    SKU_MAP[["sku", "product_id", "product"]], on="sku", how="left", validate="one_to_one")
catalog = catalog.rename(columns={"category": "category_raw"})
catalog["unit_cost"] = pd.to_numeric(catalog["unit_cost"])
assert catalog["product_id"].notna().all()
log_decision("product_catalog.csv", "Fragmented SKUs / conflicting attributes", "Stripped and upper-cased SKUs to match the POS file "
             "and attached product_id (one_to_one); costs, pack sizes and suppliers kept but not used", len(catalog),
             "Within one product the catalog contradicts itself (costs up to 27% apart, Pop-Tarts listed under Roku Inc), "
             "so it is used only as a SKU list (Q2)", len(catalog))

pd.Series({
    "rows": len(sales), "stores": sales["store_id"].nunique(), "SKUs": sales["sku"].nunique(),
    "products": sales["product_id"].nunique(), "categories": sales["category"].nunique(),
    "returns (kept, flagged)": int(sales["is_return"].sum()),
    "no timestamp (kept, flagged)": int(sales["timestamp_missing"].sum()),
    "rows used for by-date analysis": len(dated_sales),
}).to_frame("clean table")

### 2.3 Decision log

In [ ]:
with pd.option_context("display.max_colwidth", 150):
    display(pd.DataFrame(decision_log)[["file", "issue", "decision", "rows_affected", "reason", "rows_after"]])

### 2.4 Daily order volume
A sanity check before any question: what does a normal day look like, and when does the data start moving?

In [ ]:
daily_orders = dated_sales.groupby("date")["transaction_id"].nunique()

fig, ax = plt.subplots()
ax.plot(daily_orders.index, daily_orders.values, color=PALETTE[0], marker="o", ms=3.5)
ax.set_title("Transactions stepped up on Sept 16 and Sept 21, then peaked Sept 24–25")
ax.set_ylabel("Transactions per day (12 stores)")
ax.set_xlabel("Date (September 2024, store-local)")
ax.set_ylim(0)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
shade_windows(ax)
ax.legend(handles=[plt.Line2D([], [], color=PALETTE[0], marker="o", ms=3.5),
                   plt.Rectangle((0, 0), 1, 1, color=MUTED, alpha=0.25)],
          labels=["Transactions per day", "Baseline window (Sept 1–14)"], loc="best")
plt.tight_layout(); plt.show()

## 3. Weather API: Open-Meteo historical archive
**Source:** `https://archive-api.open-meteo.com/v1/archive`

**Geocoding source:** `https://geocoding-api.open-meteo.com/v1/search` (Open-Meteo Geocoding API, used for coordinates missing from the store and DC files)

**Timezone handling.** We request hourly data **in GMT on purpose** (`timezone=GMT`), the way the case's weather feed
arrived (Walmart Case, Exhibit 3). Then we convert each store's hours to that store's own local time before rolling up to calendar days. Sales
timestamps are local store time, so a storm hour at 02:00 GMT on Sept 27 is 10 pm EDT on **Sept 26** in Orlando, and 9 pm CDT
in Pensacola. Rolling up in GMT would push late-evening weather into the wrong sales day.

In [ ]:
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_VARS = ["precipitation", "wind_speed_10m", "wind_gusts_10m", "pressure_msl"]
WEATHER_CACHE = os.path.join(DATA_DIR, "weather_hourly_gmt_cache.csv")

def fetch_hourly_weather(lat, lon, start, end, retries=4, backoff=2.0, timeout=30):
    """One Open-Meteo archive call with retries and exponential backoff. Returns hourly rows with a GMT timestamp."""
    params = {
        "latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
        "hourly": ",".join(HOURLY_VARS), "timezone": "GMT",
        "wind_speed_unit": "mph", "precipitation_unit": "inch",
    }
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(ARCHIVE_URL, params=params, timeout=timeout)
            if r.status_code == 429 or r.status_code >= 500:
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            payload = r.json()
            break
        except (requests.RequestException, ValueError) as e:
            if attempt == retries:
                raise
            wait = backoff ** attempt
            print(f"  attempt {attempt} failed ({e}); retrying in {wait:.0f}s")
            time.sleep(wait)
    assert payload.get("utc_offset_seconds", 0) == 0, "expected GMT timestamps from the API"
    h = pd.DataFrame(payload["hourly"])
    h["time_gmt"] = pd.to_datetime(h.pop("time")).dt.tz_localize("UTC")
    h["source"] = f"open-meteo archive ({ARCHIVE_URL})"
    return h

def get_store_weather(stores, start="2024-08-31", end="2024-10-01"):
    """Hourly GMT weather for every store. One day of padding each side so local days at the edges are complete."""
    frames = []
    try:
        for s in stores.itertuples():
            h = fetch_hourly_weather(s.latitude, s.longitude, start, end)
            h["store_id"] = s.store_id
            frames.append(h)
            time.sleep(0.3)
        hourly = pd.concat(frames, ignore_index=True)
        hourly.to_csv(WEATHER_CACHE, index=False)   # cache of a real call, used only if a later run can't reach the API
    except Exception as e:
        if not os.path.exists(WEATHER_CACHE):
            raise
        print(f"API unavailable ({e}); loading cache of a previous successful call: {WEATHER_CACHE}")
        hourly = pd.read_csv(WEATHER_CACHE)
        hourly["time_gmt"] = pd.to_datetime(hourly["time_gmt"], utc=True)
        hourly["source"] = hourly["source"] + " [cached]"
    return hourly

def hourly_to_local_daily(hourly, stores):
    """Convert GMT hours to each store's local time, then roll up to local calendar days."""
    h = hourly.merge(stores[["store_id", "tz"]], on="store_id", how="left", validate="many_to_one")
    h["time_local"] = [t.tz_convert(z).tz_localize(None) for t, z in zip(h["time_gmt"], h["tz"])]
    h["date"] = h["time_local"].dt.normalize()
    d = (h.groupby(["store_id", "date"])
           .agg(precip_in=("precipitation", "sum"),
                max_wind_mph=("wind_speed_10m", "max"),
                max_gust_mph=("wind_gusts_10m", "max"),
                min_pressure_hpa=("pressure_msl", "min"),
                hours=("time_local", "size"),
                source=("source", "first"))
           .reset_index())
    d = d[d["date"].between("2024-09-01", "2024-09-30")]
    assert (d["hours"] >= 23).all() and (d["hours"] <= 25).all(), "incomplete local days after timezone conversion"
    return d.reset_index(drop=True)

weather_hourly = get_store_weather(stores)
weather_daily = hourly_to_local_daily(weather_hourly, stores)
print(f"hourly rows: {len(weather_hourly):,} | daily rows: {len(weather_daily)} (expected 12 stores x 30 days = 360)")
weather_daily.head()

In [ ]:
# Show the timezone shift with one real hour: the same GMT hour lands on different local dates.
ex = weather_hourly[(weather_hourly["time_gmt"] == pd.Timestamp("2024-09-27 02:00", tz="UTC")) &
                    weather_hourly["store_id"].isin(["FL-239", "FL-710"])].merge(stores[["store_id", "city", "tz"]], on="store_id")
ex["local_time"] = [t.tz_convert(z) for t, z in zip(ex["time_gmt"], ex["tz"])]
ex[["store_id", "city", "time_gmt", "local_time", "wind_gusts_10m"]]

### 3.1 The checked join: daily sales × daily weather
Grain is **store × local date**. We build the full 12 × 30 grid first, so a store-day with zero sales still appears as 0
and doesn't silently disappear. Then we check the row count, the key coverage, and the nulls.

In [ ]:
grid = pd.MultiIndex.from_product([sorted(stores["store_id"]), pd.date_range("2024-09-01", "2024-09-30")],
                                  names=["store_id", "date"]).to_frame(index=False)
store_day = (dated_sales.groupby(["store_id", "date"])
             .agg(units=("quantity", "sum"), transactions=("transaction_id", "nunique"), revenue=("revenue", "sum"))
             .reset_index())
store_day = grid.merge(store_day, on=["store_id", "date"], how="left").fillna({"units": 0, "transactions": 0, "revenue": 0})

joined = store_day.merge(weather_daily, on=["store_id", "date"], how="outer", validate="one_to_one", indicator=True)
check = {
    "sales rows": len(store_day), "weather rows": len(weather_daily), "joined rows": len(joined),
    "matched both": int((joined["_merge"] == "both").sum()),
    "sales only": int((joined["_merge"] == "left_only").sum()),
    "weather only": int((joined["_merge"] == "right_only").sum()),
    "null weather values": int(joined[["precip_in", "max_gust_mph", "min_pressure_hpa"]].isna().sum().sum()),
    "units preserved": bool(np.isclose(joined["units"].sum(), dated_sales["quantity"].sum())),
}
assert check["joined rows"] == check["matched both"] == 360, check
assert check["null weather values"] == 0 and check["units preserved"], check
joined = joined.drop(columns="_merge")
pd.Series(check).to_frame("join check")

### 3.2 Does the weather data agree with the given landfall date?
The project brief fixes landfall at **Sept 26, 2024** (`LANDFALL`, set in section 0). As a check, we find the day the
store-average max wind gust peaks (`STORM_ARRIVAL`) and the day pressure bottoms out; both should land on Sept 26.

We also date when the weather itself first turned abnormal (**`WEATHER_ALARM`**): the first day after the baseline when
the store-average max gust clears the baseline mean + 3 SD, the same rule used for sales in Q1. Q4 uses it to show what
happens if the supply chain waits for the weather.


In [ ]:
statewide_wx = joined.groupby("date")[["max_gust_mph", "precip_in", "min_pressure_hpa"]].mean()
STORM_ARRIVAL = statewide_wx["max_gust_mph"].idxmax()
gust_base = statewide_wx.loc[BASELINE_START:BASELINE_END, "max_gust_mph"]
gust_after = statewide_wx.loc[SURGE_SEARCH_START:, "max_gust_mph"]
WEATHER_ALARM = gust_after[gust_after > gust_base.mean() + 3 * gust_base.std()].index.min()
print(f"Storm arrival (peak store-average gust): {STORM_ARRIVAL:%b %d}, {statewide_wx['max_gust_mph'].max():.0f} mph | "
      f"lowest pressure: {statewide_wx['min_pressure_hpa'].idxmin():%b %d}, {statewide_wx['min_pressure_hpa'].min():.0f} hPa")
print(f"Weather alarm (first day gusts clear baseline mean + 3 SD = {gust_base.mean() + 3 * gust_base.std():.1f} mph): "
      f"{WEATHER_ALARM:%b %d}")
print(f"Given landfall date: {LANDFALL:%b %d} | weather data's peak-gust day: {STORM_ARRIVAL:%b %d} -> "
      f"{'agrees' if STORM_ARRIVAL == LANDFALL else 'DIFFERS'}")
assert abs((STORM_ARRIVAL - LANDFALL).days) <= 1, "weather data disagrees with the given landfall date"
statewide_wx.loc["2024-09-14":"2024-09-30"].round(2)

## 4. Q1: Demand surge
**Question.** Which categories surged before the storm, and by how much?

**Landfall and surge window (from the brief).** Landfall is **Sept 26, 2024**, and the brief specifies a **7-day
pre-landfall surge period**, so the surge window is **Sept 19–25** (`SURGE_START`–`SURGE_END`, set in section 0).
Landfall day itself is left out: NHC puts landfall at 11:10 pm EDT (section 7.1), and stores were closing ahead of it.
The window runs Thursday to Wednesday, so each day of the week appears exactly once.

**Baseline: Sept 1–14 (our choice).** The brief gives no baseline dates, so we picked the window that best represents
normal demand, and we can defend it on four counts:
- It is **two full weeks, Sunday to Saturday**, so each day of the week appears exactly twice and weekend shopping doesn't bias it.
- It **ends 12 days before landfall**, while gusts and pressure are normal (section 3.2).
- It is **flat**: the coefficient of variation is computed below.
- It **ends before demand broke out**. Using the same mean + 3 SD rule, we date the first abnormal day (`DEMAND_ONSET`)
  and show that the baseline stops before it. It is *not* simply the 7 days before the surge window: those days (Sept 12–18)
  already include the start of the build-up.

**Relative vs. absolute.** We rank categories by **relative** lift (% change vs. baseline), and show the **absolute**
increase (extra units per day) and its rank next to it, because the two rankings answer different questions: which
category changed the most, and which needs the most extra stock.


In [ ]:
daily_units = dated_sales.groupby("date")["quantity"].sum()
base = daily_units[BASELINE_START:BASELINE_END]
threshold = base.mean() + 3 * base.std()

# The surge window is fixed by the brief: the 7 days before landfall
assert (SURGE_END - SURGE_START).days + 1 == SURGE_DAYS and SURGE_END == LANDFALL - pd.Timedelta(days=1)
in_window = daily_units[SURGE_START:SURGE_END]

# When did demand first break out? Same mean + 3 SD rule; checks the baseline stops before the build-up
pre = daily_units[SURGE_SEARCH_START:SURGE_END]
DEMAND_ONSET = pre[pre > threshold].index.min()
assert DEMAND_ONSET > BASELINE_END, "the baseline overlaps abnormal demand"

print(f"Baseline: {BASELINE_START:%b %d}-{BASELINE_END:%b %d}, mean {base.mean():.0f} units/day, "
      f"CV {base.std() / base.mean():.1%}, threshold (mean + 3 SD) = {threshold:.0f}")
print(f"Surge window (brief: 7 days before the {LANDFALL:%b %d} landfall): {SURGE_START:%b %d}-{SURGE_END:%b %d}, "
      f"mean {in_window.mean():.0f} units/day; days above the threshold: {(in_window > threshold).sum()} of {SURGE_DAYS}")
print(f"Demand first broke out on {DEMAND_ONSET:%b %d}, {(SURGE_START - DEMAND_ONSET).days} days before the window opens "
      f"({daily_units[DEMAND_ONSET:SURGE_START - pd.Timedelta(days=1)].mean():.0f} units/day on those days)")
print(f"Landfall day ({LANDFALL:%b %d}, not in the window): {daily_units[LANDFALL]:.0f} units")
print(f"Left out of Q1 for missing timestamps: {demand['timestamp_missing'].sum()} sale rows "
      f"({demand['timestamp_missing'].mean():.1%})")


In [ ]:
def window_rate(df, start, end, by):
    """Average units per day over a window, by a grouping column."""
    n_days = (end - start).days + 1
    w = df[(df["date"] >= start) & (df["date"] <= end)]
    return w.groupby(by)["quantity"].sum() / n_days

def lift_table(df, by, base=(BASELINE_START, BASELINE_END), surge=None):
    surge = surge or (SURGE_START, SURGE_END)
    t = pd.DataFrame({"baseline_units_per_day": window_rate(df, *base, by),
                      "surge_units_per_day": window_rate(df, *surge, by)})
    t["extra_units_per_day"] = t["surge_units_per_day"] - t["baseline_units_per_day"]
    t["lift_pct"] = 100 * (t["surge_units_per_day"] / t["baseline_units_per_day"] - 1)
    t["rank_relative"] = t["lift_pct"].rank(ascending=False, method="min").astype(int)              # % change
    t["rank_absolute"] = t["extra_units_per_day"].rank(ascending=False, method="min").astype(int)  # extra units/day
    return t.sort_values("lift_pct", ascending=False).round(1)

cat_lift = lift_table(dated_sales, "category")
top5 = cat_lift.head(5).index.tolist()
moved = cat_lift[cat_lift["rank_relative"] != cat_lift["rank_absolute"]]
print(f"Ranked by % lift, #1 is {cat_lift.index[0]}; ranked by extra units/day, #1 is {cat_lift['extra_units_per_day'].idxmax()}. "
      f"{len(moved)} of {len(cat_lift)} categories change rank between the two.")
cat_lift

In [ ]:
PHASES = {
    "before the window (Sep 16-18)": ("2024-09-16", "2024-09-18"),
    "window: build (Sep 19-23)": ("2024-09-19", "2024-09-23"),
    "window: peak (Sep 24-25)": ("2024-09-24", "2024-09-25"),
    "landfall day (Sep 26)": ("2024-09-26", "2024-09-26"),
    "after (Sep 27-28)": ("2024-09-27", "2024-09-28"),
}
base_rate = window_rate(dated_sales, BASELINE_START, BASELINE_END, "category")
phase_lift = pd.DataFrame({
    name: 100 * (window_rate(dated_sales, pd.Timestamp(a), pd.Timestamp(b), "category") / base_rate - 1)
    for name, (a, b) in PHASES.items()
})
total_phase = {name: 100 * (daily_units[a:b].mean() / base.mean() - 1) for name, (a, b) in PHASES.items()}
phase_lift.loc["ALL CATEGORIES"] = total_phase
print("Lift vs. Sept 1-14 baseline, % (peak multiple = 1 + lift/100)")
phase_lift.sort_values("window: peak (Sep 24-25)", ascending=False).round(0)


In [ ]:
# Robustness 1: does the ranking depend on the baseline we chose?
alt = {
    "Sept 1-14 (chosen)": (BASELINE_START, BASELINE_END),
    "Sept 8-14 (last clean week)": (pd.Timestamp("2024-09-08"), BASELINE_END),
    "Sept 1-15 (everything before demand broke out)": (BASELINE_START, DEMAND_ONSET - pd.Timedelta(days=1)),
}
sens = pd.DataFrame({k: lift_table(dated_sales, "category", base=v)["lift_pct"] for k, v in alt.items()})
sens["rank_chosen"] = sens.iloc[:, 0].rank(ascending=False).astype(int)
print("Top 5 under each baseline:")
for k in alt:
    print(f"  {k:<48}", sens[k].sort_values(ascending=False).head(5).index.tolist())

# Robustness 2: does it depend on the window? Brief's 7 days vs. the whole build-up (first abnormal day to landfall eve)
win_alt = lift_table(dated_sales, "category", surge=(DEMAND_ONSET, SURGE_END))["lift_pct"]
print(f"\nTop 5, brief's 7-day window ({SURGE_START:%b %d}-{SURGE_END:%b %d}):", cat_lift.head(5).index.tolist())
print(f"Top 5, whole build-up ({DEMAND_ONSET:%b %d}-{SURGE_END:%b %d}):      ", win_alt.sort_values(ascending=False).head(5).index.tolist())
sens.assign(**{f"window {DEMAND_ONSET:%b %d}-{SURGE_END:%b %d}": win_alt}).round(0)


In [ ]:
# Missing timestamps: are they spread evenly? If they cluster in one category, Q1 would be biased.
miss_share = pd.DataFrame({
    "share_of_all_sales": demand.groupby("category")["quantity"].sum() / demand["quantity"].sum(),
    "share_of_undated_sales": demand[demand["timestamp_missing"]].groupby("category")["quantity"].sum() / demand.loc[demand["timestamp_missing"], "quantity"].sum(),
}).mul(100).round(1)
miss_share["gap_pts"] = (miss_share["share_of_undated_sales"] - miss_share["share_of_all_sales"]).round(1)
miss_share.sort_values("gap_pts")

In [ ]:
# Chart: baseline vs surge, top 5 categories by lift
plot = cat_lift.loc[top5].iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 4.6))
y = np.arange(len(plot))
ax.barh(y + 0.2, plot["baseline_units_per_day"], height=0.38, color=MUTED, label="Baseline (Sept 1–14)")
ax.barh(y - 0.2, plot["surge_units_per_day"], height=0.38, color=PALETTE[1],
        label=f"Surge window ({SURGE_START:%b %d}–{SURGE_END:%b %d})")
for yi, (_, r) in zip(y, plot.iterrows()):
    ax.text(r["surge_units_per_day"] + 3, yi - 0.2, f"+{r['lift_pct']:.0f}%", va="center", fontsize=10, color="#0b0b0b")
ax.set_yticks(y, plot.index)
ax.set_xlabel("Average units sold per day, all 12 stores")
ax.set_title(f"{top5[0]} had the biggest lift; {cat_lift['extra_units_per_day'].idxmax()} added the most units")
ax.grid(axis="y", visible=False)
ax.set_ylabel("Category")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
# Chart: daily units for the top 5 categories over the month
daily_cat = dated_sales.groupby(["date", "category"])["quantity"].sum().unstack().fillna(0)
fig, ax = plt.subplots(figsize=(11, 5))
for i, c in enumerate(top5):
    ax.plot(daily_cat.index, daily_cat[c], color=PALETTE[i], label=c)
ax.set_title(f"Demand broke out {DEMAND_ONSET:%b %d}, climbed through the 7-day window (shaded), peaked Sept 24–25, fell at landfall")
ax.set_ylabel("Units sold per day (12 stores)")
ax.set_xlabel("Date (September 2024)")
ax.set_ylim(0)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.axvspan(SURGE_START, SURGE_END + pd.Timedelta(days=1), color=PALETTE[1], alpha=0.08, lw=0)
shade_windows(ax, arrival=LANDFALL)
ax.legend(loc="upper left", ncol=5, bbox_to_anchor=(0, -0.12))
plt.tight_layout(); plt.show()

In [ ]:
q1 = cat_lift
print(f"Q1 numbers: surge window {SURGE_START:%b %d}–{SURGE_END:%b %d} (7 days before landfall); total units/day "
      f"{base.mean():.0f} -> {daily_units[SURGE_START:SURGE_END].mean():.0f} "
      f"(+{100 * (daily_units[SURGE_START:SURGE_END].mean() / base.mean() - 1):.0f}%)")
for c, r in q1.head(5).iterrows():
    print(f"  {c:<14} {r.baseline_units_per_day:>6.1f} -> {r.surge_units_per_day:>6.1f} units/day (+{r.lift_pct:.0f}%, "
          f"+{r.extra_units_per_day:.0f} units/day, absolute rank #{int(r.rank_absolute)})")
print(f"Smallest lift: {q1.index[-1]} (+{q1['lift_pct'].iloc[-1]:.0f}%)")
print(f"Biggest absolute increase: {q1['extra_units_per_day'].idxmax()} (+{q1['extra_units_per_day'].max():.0f} units/day)")
for name, (a, b) in PHASES.items():
    print(f"  {name:<30} total {phase_lift.loc['ALL CATEGORIES', name]:+.0f}%")
print("Peak (Sep 24-25) multiple, Breakfast / Emergency:",
      f"{1 + phase_lift.loc['Breakfast', 'window: peak (Sep 24-25)'] / 100:.1f}x /",
      f"{1 + phase_lift.loc['Emergency', 'window: peak (Sep 24-25)'] / 100:.1f}x")

### Q1 findings
*Every number here is printed by the cells above.*

**Window and baseline.** Landfall is Sept 26, and the brief's surge period is the **7 days before it: Sept 19–25**.
Our baseline, **Sept 1–14**, is flat: about **1,109 units/day** with a coefficient of variation of 2.9%. All 7 days of
the surge window clear the mean + 3 SD threshold of 1,206 units/day. Over the window, total volume averaged
**2,712 units/day (+145%)**.

**Demand started moving before the window.** By the same rule, demand first broke out on **Sept 16**, three days before
the brief's window opens, so the baseline stops well before any build-up. The phases:
- **Before the window (Sept 16–18): +38%.** The weather data still shows normal gusts and pressure here (section 3.2).
- **Window, build (Sept 19–23): +101%.**
- **Window, peak (Sept 24–25): +254%**, about 3.5× normal.
- **Landfall day (Sept 26), outside the window: +101%**, already falling.
- **After (Sept 27–28): +53%.** Sept 29–30 are back to a normal ~1,070–1,080 units/day.

**Relative vs. absolute: the two rankings disagree, so we show both.** 8 of 10 categories change rank between them.

| Category | Baseline → window (units/day) | **Relative lift** (rank) | **Absolute increase** (rank) |
|---|---|---|---|
| Breakfast | 74 → 377 | **+410%** (#1) | +303/day (#2) |
| Emergency | 357 → 1,006 | **+182%** (#2) | **+650/day (#1)** |
| Electronics | 70 → 176 | +152% (#3) | +106/day (#5) |
| Alcohol | 62 → 139 | +126% (#4) | +77/day (#7) |
| Health | 42 → 95 | +124% (#5) | +53/day (#8) |
| Entertainment | 103 → 229 | +123% (#6) | +126/day (#4) |
| Grocery | 172 → 307 | +78% (#8) | **+134/day (#3)** |

- **Breakfast is entirely Strawberry Pop-Tarts.** At the Sept 24–25 peak it hit **8.4× normal**, close to the Pop-Tarts
  jump in the *Walmart Case* (Exhibit 2: +600%, or 7×).
- **Emergency** (water, batteries, flashlights, candles, tarps) has the second-highest lift but by far the most volume:
  **+650 extra units a day**, about 40% of all extra units. At the peak it ran at 4.1× normal.
- **Grocery** shows why the distinction matters: only #8 by lift, but #3 by extra units.
- **Snacks** barely moved (+24%), a useful control: not everything goes up.

**Is this robust?** Breakfast, Emergency and Electronics are #1–3 under all three baselines and under the longer
Sept 16–25 window. Alcohol, Health and Entertainment sit within 3 points of each other at #4–6, so treat those three as
a tie. The 3.5% of sales with no timestamp fall across categories within 2 points of each category's share of all
sales, so leaving them out doesn't bias the ranking.

**For leadership:** the buying window opens about **10 days before landfall** and peaks one to two days before it.
Emergency needs the most trucks; Pop-Tarts needs the biggest relative increase, about 8× at the peak.

## 5. Q2: SKU consolidation
**Question.** How much does SKU fragmentation hide, and what does the picture look like once SKUs are consolidated?

The mapping table (`SKU_MAP`, section 2.1) maps 45 SKUs to 20 products. Below: the mapping, then before vs. after.

In [ ]:
SKU_MAP.groupby(["product_id", "product", "category"])["sku"].agg(lambda s: ", ".join(s)).reset_index().rename(columns={"sku": "SKUs merged"})

In [ ]:
# Catalog conflicts inside the same product: these are why the catalog can't be trusted for cost or pack size
conf = catalog.groupby("product").agg(
    skus=("sku", "count"),
    names=("product_name", "nunique"),
    raw_categories=("category_raw", "nunique"),
    unit_cost_min=("unit_cost", "min"), unit_cost_max=("unit_cost", "max"),
    package_sizes=("package_size", lambda s: ", ".join(sorted(s.dropna().unique()))),
    suppliers=("supplier", lambda s: ", ".join(sorted(s.dropna().unique()))),
)
conf["cost_spread_pct"] = (100 * (conf["unit_cost_max"] / conf["unit_cost_min"] - 1)).round(0)
conf.sort_values("skus", ascending=False).head(8)

In [ ]:
surge_sales = dated_sales[(dated_sales["date"] >= SURGE_START) & (dated_sales["date"] <= SURGE_END)]
n_surge_days = (SURGE_END - SURGE_START).days + 1

before = (surge_sales.groupby(["sku", "product_name"])["quantity"].sum() / n_surge_days).rename("units_per_day").reset_index()
before["rank"] = before["units_per_day"].rank(ascending=False, method="min").astype(int)
after_ = (surge_sales.groupby("product_name")["quantity"].sum() / n_surge_days).rename("units_per_day").reset_index()
after_["rank"] = after_["units_per_day"].rank(ascending=False, method="min").astype(int)

# Consolidation re-labels sales, it never adds them: the totals must match exactly
totals = pd.Series({"fragmented (sum over 45 SKUs)": before["units_per_day"].sum(),
                    "consolidated (sum over 20 products)": after_["units_per_day"].sum()})
assert np.isclose(*totals), "SKU consolidation created or lost sales"
print("Surge-window units/day, all lines:", totals.round(1).to_dict(), "-> identical")

pt = "Pop-Tarts Strawberry"
pt_before = before[before["product_name"] == pt].sort_values("rank")
pt_after = after_[after_["product_name"] == pt].iloc[0]
summary = pd.DataFrame({
    "before (by SKU)": [before["sku"].nunique(), int(pt_before["rank"].min()), round(pt_before["units_per_day"].max(), 1),
                        f"{len(pt_before)} separate lines"],
    "after (by product)": [after_["product_name"].nunique(), int(pt_after["rank"]), round(pt_after["units_per_day"], 1),
                           "1 line"],
}, index=["items a buyer has to scan", "best Pop-Tarts rank (surge window)", "Pop-Tarts units/day (largest line)",
          "how Pop-Tarts shows up"])
summary

In [ ]:
# Lift before and after: does fragmentation change the % story, the volume story, or both?
sku_lift = lift_table(dated_sales, "sku")
prod_lift = lift_table(dated_sales, "product_name")
pt_skus = SKU_MAP.loc[SKU_MAP["product"] == pt, "sku"]
cmp = pd.concat([sku_lift.loc[pt_skus].assign(level="SKU"),
                 prod_lift.loc[[pt]].assign(level="Product")])
cmp

In [ ]:
# Chart: top 10 lines in the surge window, by SKU vs. by product
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharex=True)
b10 = before.nlargest(10, "units_per_day").iloc[::-1]
a10 = after_.nlargest(10, "units_per_day").iloc[::-1]
for ax, d, label, title in [(axes[0], b10, "sku", "Before: by SKU"), (axes[1], a10, "product_name", "After: by product")]:
    colors = [PALETTE[1] if (n == pt) else PALETTE[0] for n in d["product_name"]]
    ax.barh(d[label], d["units_per_day"], color=colors, height=0.7)
    ax.set_title(title, fontsize=11.5)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Units per day in the surge window")
ax.set_ylabel(label.replace("_", " "))
axes[1].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=PALETTE[1]), plt.Rectangle((0, 0), 1, 1, color=PALETTE[0])],
               labels=["Pop-Tarts Strawberry", "every other product"], loc="lower right")
fig.suptitle(f"Split four ways, each Pop-Tarts SKU looks like one flashlight SKU; consolidated, it is the #{int(pt_after['rank'])} product (orange)",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Chart: product-level lift after consolidation, all 20 products
pl = prod_lift.sort_values("lift_pct")
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.barh(pl.index, pl["lift_pct"], color=[PALETTE[1] if p == pt else PALETTE[0] for p in pl.index], height=0.7)
for i, (p, v) in enumerate(pl["lift_pct"].items()):
    ax.text(v + 5, i, f"+{v:.0f}%", va="center", fontsize=8.5, color="#52514e")
ax.set_title("Consolidated, Pop-Tarts has the highest lift of all 20 products, about double the next one")
ax.set_xlabel("Lift vs. Sept 1–14 baseline (%)")
ax.set_ylabel("Product (after SKU consolidation)")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.grid(axis="y", visible=False)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=PALETTE[1]), plt.Rectangle((0, 0), 1, 1, color=PALETTE[0])],
          labels=["Pop-Tarts Strawberry", "every other product"], loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
pt_lift = prod_lift.loc[pt, "lift_pct"]
print(f"Pop-Tarts: {len(pt_skus)} SKUs -> 1 product | product lift +{pt_lift:.0f}% | "
      f"rank by surge volume {int(pt_before['rank'].min())} (best single SKU) -> {int(pt_after['rank'])} (consolidated)")
print("Product lift ranking (top 6):", prod_lift.head(6)["lift_pct"].to_dict())

### Q2 findings

**The mapping table.** `SKU_MAP` maps **45 SKUs to 20 products**, one row per SKU, linking SKUs that cashiers sold under the
same product name. A left join confirms every SKU in the sales data is covered.
- **Strawberry Pop-Tarts** was split across four SKUs: `PT-12`, `SKU#459812`, `POPTART-STRAW` and `PT_STRAWBERRY`.
- **Bottled water, AA batteries and Monopoly** were each split across three.
- Every other product was split across two.

**Consolidation creates no sales.** Same rows, same units and same revenue before and after the merge (asserted in 2.2).
In the surge window, the 45 SKU lines and the 20 product lines both add up to **2,711.9 units/day**.

**Fragmented vs. consolidated (Sept 19–25 surge window).**
- **By SKU (fragmented):** the top two lines are bottled water (~107–109 units/day each). The four Pop-Tarts SKUs sell
  ~90–103 units/day each and sit at **ranks 3, 5, 6 and 7**, looking like four ordinary items.
- **By product (consolidated):** Pop-Tarts is the **#1 product at 377 units/day**, ahead of water (320) and batteries (243).

**What changes and what doesn't.** Consolidation leaves the **percentage lift** almost unchanged: each Pop-Tarts SKU shows
+400% to +433%, and the product shows +410%. What it fixes is **size and rank**. A buyer reading a SKU report sees four
ordinary lines, with water on top. In fact Pop-Tarts is the biggest-moving product before the storm, and its lift is the
highest of all 20 products, **about double the next one** (bottled water, +195%).

**The catalog can't be trusted for anything but the SKU list.** Within the same product:
- **Unit costs** differ by up to 27% (water) and 26% (Monopoly).
- **Package sizes** contradict each other: water is listed as "1 pack" and "24 pack", batteries as "4pk" and "24 pack".
- **Suppliers** are implausible: Pop-Tarts is listed under Roku Inc and Johnson & Johnson.

That's why we consolidated on POS evidence. Any cost or margin analysis should wait for a catalog fix.

**Recommendation:** forecast and replenish at the **product** level, and keep the SKU→product map (`SKU_MAP`) as a
maintained ETL step.

## 6. Q3: Weather correlation and lag
**Question.** Does the storm data actually line up with the sales pattern, and does buying *lead* the weather?

We use the checked store × day table from section 3.1 (360 rows, GMT weather already converted to each store's local day).
The weather variables are daily **max wind gust (mph)**, **total precipitation (in)** and **minimum sea-level pressure (hPa)**.
Pressure is the cleanest storm-proximity signal: it falls as the storm gets closer. Two views:
1. **Same-day correlation**, pooled across store-days and statewide (daily totals/averages across the 12 stores).
2. **Lag analysis:** correlate sales on day *t* with weather on day *t + k*, inside each store, then average across stores.
   A peak at **k > 0** means sales **lead** the weather: people buy *before* the storm arrives.

In [ ]:
WX = ["max_gust_mph", "precip_in", "min_pressure_hpa"]
statewide = joined.groupby("date").agg(units=("units", "sum"), **{c: (c, "mean") for c in WX})

same_day = pd.DataFrame({
    "pooled r (360 store-days)": [joined["units"].corr(joined[c]) for c in WX],
    "pooled Spearman": [joined["units"].corr(joined[c], method="spearman") for c in WX],
    "statewide r (30 days)": [statewide["units"].corr(statewide[c]) for c in WX],
}, index=WX).round(2)
same_day

In [ ]:
LAGS = range(-5, 6)

def lag_corr(df, var, lags=LAGS):
    # corr(units on day t, weather on day t+k) inside each store, then averaged across stores
    out = {}
    for k in lags:
        rs = []
        for _, g in df.sort_values("date").groupby("store_id"):
            rs.append(g["units"].corr(g[var].shift(-k)))
        out[k] = np.nanmean(rs)
    return pd.Series(out, name=var)

lags = pd.concat([lag_corr(joined, v) for v in WX], axis=1).round(2)
lags.index.name = "k (weather is k days after sales)"
best = {v: (int(lags[v].abs().idxmax()), lags[v].loc[lags[v].abs().idxmax()]) for v in WX}
print("Strongest lag per variable:", {v: f"k={k:+d}, r={r:+.2f}" for v, (k, r) in best.items()})
lags

In [ ]:
# Chart: statewide sales and weather on two stacked panels with a shared date axis (no dual y-axis)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6.6), sharex=True)
a1.bar(statewide.index, statewide["units"], color=PALETTE[0], width=0.8, label="Units sold, all 12 stores")
a1.set_ylabel("Units sold per day")
a1.set_title("Sales built for 10 calm, dry days and peaked the day before the wind arrived")
a1.legend(loc="upper left")
a2.plot(statewide.index, statewide["max_gust_mph"], color=PALETTE[1], marker="o", ms=3.5, label="Max wind gust (store average)")
a2.set_ylabel("Max gust (mph)")
a2.set_ylim(0)
for ax in (a1, a2):
    ax.axvline(LANDFALL + pd.Timedelta(hours=12), color="#52514e", ls="--", lw=1)
a2.text(LANDFALL + pd.Timedelta(hours=18), a2.get_ylim()[1] * 0.95, "landfall", ha="left", va="top", fontsize=9, color="#52514e")
a2.set_xlabel("Date (September 2024, store-local)")
a2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
a2.legend(loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
# Chart: lag correlation profile
fig, ax = plt.subplots(figsize=(10, 4.6))
w = 0.27
labels = {"max_gust_mph": "Max gust", "precip_in": "Precipitation", "min_pressure_hpa": "Min pressure"}
for i, v in enumerate(WX):
    ax.bar(np.array(list(LAGS)) + (i - 1) * w, lags[v], width=w, color=PALETTE[i], label=labels[v])
ax.axhline(0, color="#52514e", lw=0.8)
ax.axvline(0, color=MUTED, lw=0.8, ls=":")
ax.set_xticks(list(LAGS))
ax.set_xlabel("k: weather measured k days AFTER the sales day  (k > 0 means sales lead the weather)")
ax.set_ylabel("Avg within-store correlation r")
k_g, r_g = best["max_gust_mph"]; k_p, r_p = best["min_pressure_hpa"]
ax.set_title(f"Sales track gusts and pressure {k_g} days later (r = {r_g:+.2f} and {r_p:+.2f}), not the same day")
ax.grid(axis="x", visible=False)
ax.legend(loc="lower left", ncol=3)
plt.tight_layout(); plt.show()

In [ ]:
# Chart: Emergency units vs. max gust k* days later, every store-day
k_star, r_star = best["max_gust_mph"]
em = (dated_sales[dated_sales["category"] == "Emergency"].groupby(["store_id", "date"])["quantity"].sum()
      .rename("emergency_units").reset_index())
sc = joined.merge(em, on=["store_id", "date"], how="left").fillna({"emergency_units": 0}).sort_values(["store_id", "date"])
sc["gust_future"] = sc.groupby("store_id")["max_gust_mph"].shift(-k_star)
sc = sc.dropna(subset=["gust_future"])
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.scatter(sc["gust_future"], sc["emergency_units"], s=22, color=PALETTE[0], alpha=0.6, edgecolor="white", linewidth=0.6,
           label="One store-day")
slope, intercept = np.polyfit(sc["gust_future"], sc["emergency_units"], 1)
xs = np.linspace(sc["gust_future"].min(), sc["gust_future"].max(), 50)
ax.plot(xs, slope * xs + intercept, color=PALETTE[1], lw=2, label="Least-squares fit")
r_em = sc["emergency_units"].corr(sc["gust_future"])
ax.set_title(f"Emergency sales vs. max gust {k_star:+d} days later (r = {r_em:.2f}, n = {len(sc)} store-days)")
ax.set_xlabel(f"Max wind gust {k_star} days later (mph)")
ax.set_ylabel("Emergency units sold that day")
ax.set_ylim(0)
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
peak_sales_day = statewide["units"].idxmax()
peak_gust_day = statewide["max_gust_mph"].idxmax()
low_pressure_day = statewide["min_pressure_hpa"].idxmin()
print(f"Statewide peak sales day: {peak_sales_day:%b %d} | peak gust day: {peak_gust_day:%b %d} | lowest pressure day: {low_pressure_day:%b %d}")
print(f"Sales lead the gust peak by {(peak_gust_day - peak_sales_day).days} day(s)")
print("Same-day r:", same_day["pooled r (360 store-days)"].to_dict())
print("Best lag:", {v: f"k={k:+d}, r={r:+.2f}" for v, (k, r) in best.items()})
pensacola_windiest = joined[joined["store_id"] == "FL-710"].sort_values("max_gust_mph").iloc[-1]
print(f"Pensacola's windiest day: {pensacola_windiest['date']:%b %d}, {pensacola_windiest['max_gust_mph']:.0f} mph, "
      f"{pensacola_windiest['units']:.0f} units vs. a baseline mean of "
      f"{joined[(joined['store_id'] == 'FL-710') & joined['date'].between(BASELINE_START, BASELINE_END)]['units'].mean():.0f}")
top_gust = joined.loc[joined.groupby("store_id")["max_gust_mph"].idxmax(), ["store_id", "date", "max_gust_mph"]].merge(
    stores[["store_id", "city"]]).sort_values("max_gust_mph", ascending=False)
top_gust

### Q3 finding
*Every number here is printed by the cells above.*

**Same-day weather barely explains sales.** Across 360 store-days, units correlate only weakly with that day's max gust
(r = 0.18), rain (r = −0.17) and pressure (r = −0.21). The statewide table in section 3.2 shows why: the biggest surge days,
**Sept 21–24, were calm and dry**. Average max gusts were 18–21 mph, the same as the baseline. Rain was 0.01–0.03 in, and
pressure sat at a normal ~1,011 hPa.

**Weather two days ahead explains sales well.** Lining each store's sales up against its own weather **k days later**
gives a clear peak at **k = +2**:
- **Max gust:** r = **+0.71**.
- **Minimum pressure:** r = **−0.76**. Negative, because lower pressure means a closer storm.
- **Emergency category alone:** r ≈ 0.69 against gusts two days later.
- **Rain:** no relationship at any positive lead (|r| ≤ 0.06 for k ≥ 1). The only sizable rain correlations are at
  negative k (rain *before* the sales day), which just reflects a rainy early September.

**The timeline.**
- **Sales** peaked statewide on **Sept 25**.
- **Landfall Sept 26** (11:10 pm EDT per NHC, section 7.1); the weather data agrees: store-average max gust peaked that day
  at 58 mph, pressure bottomed at 996 hPa, and it was the rainiest day of the month (0.88 in).
- **Strongest gusts:** St. Petersburg 77 mph and Tampa 73 mph on Sept 26, then Gainesville 71 mph on Sept 27.
- **Sales fell** as soon as the wind arrived.

**A useful control: Pensacola.** Its windiest day of the month fell inside our baseline (Sept 12, about 47 mph), and its
sales that day were slightly *below* normal. Wind with no storm threat to the state doesn't move buying.

**Our interpretation:** customers **buy on the forecast, not on the weather**. A weather feed is useful as a *leading* signal,
read two or more days out. As a same-day trigger it arrives after demand has already peaked.

**Caveats we checked.**
- **Timezones:** the weather came in GMT and was converted per store before rolling up to days. 02:00 GMT on Sept 27 is
  10 pm Sept 26 in Orlando and 9 pm Sept 26 in Pensacola, which is on Central time (section 3).
- **The join:** exactly 360 rows, no nulls, and every dated unit preserved (section 3.1).
- **Sample size:** one storm over 30 days. These r values describe this storm; they are not a forecast model.
- **Causation:** sales and future weather both respond to the same approaching storm, so the lag is evidence of
  forecast-driven buying, not proof that weather causes sales.

## 7. Q4: Stores and geography
**Question.** Did stores nearer the storm behave differently? Did every store surge the same way, and does geography
explain the differences?

**Inputs.** The store file joins cleanly to every transaction: `clean_store_id()` collapsed 60 spellings to 12 ids, and the
join check in section 1 found no orphans. For each store we measure:
- its **lift** (surge window vs. baseline), its **peak multiple** (Sept 24–25 vs. baseline) and the **day its own surge started**;
- its **storm exposure** from the weather join: max gust and lowest pressure between Sept 25 and 28;
- its **distance from the hurricane's path**: the closest approach of the **official NHC best track** for Hurricane Helene
  (7.1 below), plus its distance from the landfall point. Store locations alone can't say which stores were closest to
  the path, so we use the historical track, and check it against the Open-Meteo weather data.

### 7.1 Where was the storm? The official NHC track, checked against the weather data
**Source.** The National Hurricane Center's best track for Hurricane Helene (storm ID **AL092024**), from the
**HURDAT2** Atlantic database (https://www.nhc.noaa.gov/data/hurdat/), the same track as the NHC Tropical Cyclone Report
(https://www.nhc.noaa.gov/data/tcr/AL092024_Helene.pdf). HURDAT2 gives the storm's center every 6 hours, plus special
fixes such as landfall (record `L`), with maximum wind (kt) and central pressure (mb). Times are GMT.

**Method.**
- Download HURDAT2 and keep Helene's rows. If NHC can't be reached, the notebook uses the same rows copied below from the
  file (Sept 24–27) and says so; when it can, it asserts that the two agree.
- Interpolate the track to 10-minute steps and measure each store's **closest approach** to it, and when that was.
- **Cross-check:** we ask Open-Meteo for hourly sea-level pressure on a 0.5° grid over Florida and the eastern Gulf
  (Sept 25–27, GMT) and compare the weather data's pressure minimum with NHC's position at the same hour.


In [ ]:
def haversine_mi(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 3958.8 * np.arcsin(np.sqrt(a))

HURDAT_URL = "https://www.nhc.noaa.gov/data/hurdat/hurdat2-1851-2024-040425.txt"   # NHC HURDAT2, through the 2024 season
TCR_URL = "https://www.nhc.noaa.gov/data/tcr/AL092024_Helene.pdf"                   # NHC report on Helene (same best track)

# Helene's HURDAT2 rows, Sept 24 00Z - Sept 27 18Z, copied from the file above. Used only if NHC can't be reached.
HELENE_HURDAT2_COPY = """\
AL092024,             HELENE,     19,
20240924, 0000,  , DB, 18.2N,  82.2W,  35, 1002
20240924, 0600,  , DB, 18.6N,  82.8W,  35, 1001
20240924, 1200,  , TS, 19.2N,  83.7W,  40,  999
20240924, 1800,  , TS, 19.4N,  84.6W,  45,  997
20240925, 0000,  , TS, 19.7N,  85.2W,  50,  990
20240925, 0600,  , TS, 20.3N,  85.9W,  55,  985
20240925, 1200,  , HU, 21.1N,  86.2W,  65,  979
20240925, 1800,  , HU, 22.0N,  86.5W,  70,  978
20240926, 0000,  , HU, 22.8N,  86.7W,  70,  973
20240926, 0600,  , HU, 23.6N,  86.5W,  75,  970
20240926, 1200,  , HU, 24.7N,  85.8W,  85,  963
20240926, 1800,  , HU, 26.6N,  85.0W, 105,  958
20240927, 0000,  , HU, 28.7N,  84.3W, 120,  941
20240927, 0310, L, HU, 30.0N,  83.7W, 120,  939
20240927, 0500, R, HU, 30.8N,  83.5W,  95,  947
20240927, 0600,  , HU, 31.3N,  83.3W,  80,  957
20240927, 0900, R, TS, 32.9N,  83.1W,  60,  967
20240927, 1200,  , TS, 34.4N,  83.2W,  45,  973
20240927, 1800,  , LO, 36.8N,  84.9W,  40,  983
"""

def get_with_retries(url, params=None, retries=4, backoff=2.0, timeout=90):
    # GET with retries and exponential backoff (429 and 5xx are retried, other errors raise)
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code == 429 or r.status_code >= 500:
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                raise
            wait = backoff ** attempt
            print(f"  attempt {attempt} failed ({e}); retrying in {wait:.0f}s")
            time.sleep(wait)

def get_json(url, params, **kw):
    return get_with_retries(url, params, **kw).json()

def parse_hurdat2(text, storm_id="AL092024"):
    """One storm's rows from HURDAT2 text -> GMT time, lat, lon (west negative), wind (kt), pressure (mb)."""
    lines = text.splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith(storm_id))
    n = int(lines[i].split(",")[2])
    rows = [[f.strip() for f in l.split(",")[:8]] for l in lines[i + 1:i + 1 + n]]
    t = pd.DataFrame(rows, columns=["date", "hhmm", "record", "status", "lat", "lon", "wind_kt", "pressure_mb"])
    t["time_gmt"] = pd.to_datetime(t["date"] + t["hhmm"], format="%Y%m%d%H%M").dt.tz_localize("UTC")
    t["lat"] = t["lat"].str[:-1].astype(float) * np.where(t["lat"].str.endswith("S"), -1, 1)
    t["lon"] = t["lon"].str[:-1].astype(float) * np.where(t["lon"].str.endswith("W"), -1, 1)
    t[["wind_kt", "pressure_mb"]] = t[["wind_kt", "pressure_mb"]].astype(int)
    return t[["time_gmt", "record", "status", "lat", "lon", "wind_kt", "pressure_mb"]]

nhc_copy = parse_hurdat2(HELENE_HURDAT2_COPY)
try:
    nhc_all = parse_hurdat2(get_with_retries(HURDAT_URL, timeout=120).text)
    nhc_all["source"] = f"NHC HURDAT2 ({HURDAT_URL})"
    # the copied rows must match the live file exactly
    chk = nhc_copy.merge(nhc_all, on="time_gmt", suffixes=("_copy", "_nhc"), validate="one_to_one")
    assert len(chk) == len(nhc_copy) and np.allclose(chk[["lat_copy", "lon_copy"]].values, chk[["lat_nhc", "lon_nhc"]].values)
    print(f"NHC HURDAT2 downloaded: {len(nhc_all)} rows for AL092024 (Helene); the copied rows match")
except Exception as e:
    print(f"NHC unavailable ({e}); using the HURDAT2 rows copied into this notebook")
    nhc_all = nhc_copy.assign(source=f"NHC HURDAT2 rows copied from {HURDAT_URL}")

# the part of the track that matters for Florida: Gulf approach through landfall and inland
nhc = nhc_all[nhc_all["time_gmt"].between("2024-09-25 00:00", "2024-09-27 18:00", inclusive="both")].reset_index(drop=True)
lf = nhc_all[(nhc_all["record"] == "L") & nhc_all["time_gmt"].between("2024-09-26", "2024-09-28")].iloc[0]
LANDFALL_TIME_LOCAL = lf["time_gmt"].tz_convert("America/New_York")
STORM_LAT, STORM_LON = float(lf["lat"]), float(lf["lon"])        # landfall point
assert LANDFALL_TIME_LOCAL.normalize().tz_localize(None) == LANDFALL, "NHC landfall date differs from the brief"
print(f"NHC landfall: {lf['time_gmt']:%b %d %H:%M} GMT = {LANDFALL_TIME_LOCAL:%b %d %I:%M %p} EDT, "
      f"{STORM_LAT:.1f}N {abs(STORM_LON):.1f}W, {lf['wind_kt']} kt, {lf['pressure_mb']} mb (matches the brief's Sept 26)")

# 10-minute interpolation along the track, for closest-approach distances
step = pd.date_range(nhc["time_gmt"].min(), nhc["time_gmt"].max(), freq="10min")
nhc_dense = (nhc.set_index("time_gmt")[["lat", "lon"]].reindex(nhc["time_gmt"].tolist() + list(step))
             .sort_index().loc[lambda d: ~d.index.duplicated()].interpolate(method="time").loc[step].reset_index()
             .rename(columns={"index": "time_gmt"}))
log_decision("NHC HURDAT2 (external)", "Hurricane path not in the project files",
             "Loaded the official best track for Helene (AL092024), Sept 25-27; interpolated to 10-minute steps; "
             "measured each store's closest approach and its distance to the landfall point", len(nhc),
             "Clarification for Q4: store locations alone don't establish which stores were closest to the path; "
             "the historical track does", len(nhc_dense))
nhc[["time_gmt", "record", "status", "lat", "lon", "wind_kt", "pressure_mb"]]


**Cross-check with the weather data.** Does the storm in Open-Meteo's pressure field sit where NHC says it was?

In [ ]:
def fetch_pressure_grid(start="2024-09-25", end="2024-09-27"):
    # Hourly GMT sea-level pressure on a lat/lon grid, one multi-location Open-Meteo call
    pts = [(la, lo) for la in GRID_LATS for lo in GRID_LONS]
    params = {"latitude": ",".join(f"{la:.2f}" for la, _ in pts), "longitude": ",".join(f"{lo:.2f}" for _, lo in pts),
              "start_date": start, "end_date": end, "hourly": "pressure_msl", "timezone": "GMT"}
    payload = get_json(ARCHIVE_URL, params)
    payload = payload if isinstance(payload, list) else [payload]
    frames = []
    for p in payload:
        assert p.get("utc_offset_seconds", 0) == 0, "expected GMT timestamps from the API"
        h = pd.DataFrame(p["hourly"])
        h["lat"], h["lon"] = p["latitude"], p["longitude"]
        frames.append(h)
    g = pd.concat(frames, ignore_index=True)
    g["time_gmt"] = pd.to_datetime(g.pop("time")).dt.tz_localize("UTC")
    g["source"] = f"open-meteo archive ({ARCHIVE_URL})"
    return g

GRID_LATS = np.arange(26.0, 31.01, 0.5)
GRID_LONS = np.arange(-87.5, -80.99, 0.5)
STORM_CACHE = os.path.join(DATA_DIR, "storm_grid_gmt_cache.csv")

try:
    grid_wx = fetch_pressure_grid()
    grid_wx.to_csv(STORM_CACHE, index=False)
except Exception as e:
    if os.path.exists(STORM_CACHE):
        print(f"API unavailable ({e}); loading cache of a previous successful call")
        grid_wx = pd.read_csv(STORM_CACHE)
        grid_wx["time_gmt"] = pd.to_datetime(grid_wx["time_gmt"], utc=True)
    else:
        print(f"Grid call failed ({e}); skipping the cross-check (Q4 uses the NHC track either way)")
        grid_wx = None

def nhc_position(t):
    """NHC center at GMT time t, interpolated along the track."""
    k = (nhc_dense["time_gmt"] - t).abs().idxmin()
    return nhc_dense.loc[k, "lat"], nhc_dense.loc[k, "lon"]

grid_track = pd.DataFrame(columns=["time_gmt", "lat", "lon", "pressure_msl"])
if grid_wx is not None:
    grid_wx = grid_wx.dropna(subset=["pressure_msl"]).reset_index(drop=True)
    on_edge = (grid_wx["lat"].round(1).isin([GRID_LATS.min(), GRID_LATS.max()]) |
               grid_wx["lon"].round(1).isin([GRID_LONS.min(), GRID_LONS.max()]))
    hourly_low = grid_wx.loc[grid_wx.groupby("time_gmt")["pressure_msl"].idxmin()]
    # keep hours where the low is inside the grid (not on its edge) and clearly a storm
    grid_track = hourly_low[~on_edge.loc[hourly_low.index] & (hourly_low["pressure_msl"] < 1000)].sort_values("time_gmt").copy()
    pos = [nhc_position(t) for t in grid_track["time_gmt"]]
    grid_track["miles_from_nhc"] = haversine_mi(grid_track["lat"].values, grid_track["lon"].values,
                                                np.array([p[0] for p in pos]), np.array([p[1] for p in pos]))
    center = grid_wx.loc[grid_wx["pressure_msl"].idxmin()]
    print(f"Open-Meteo pressure minimum: {center['pressure_msl']:.0f} hPa at {center['lat']:.2f}N {abs(center['lon']):.2f}W, "
          f"{center['time_gmt']:%b %d %H:%M} GMT, "
          f"{haversine_mi(center['lat'], center['lon'], STORM_LAT, STORM_LON):.0f} miles from NHC's landfall point")
    print(f"Hourly weather-data lows vs. NHC position at the same hour: median {grid_track['miles_from_nhc'].median():.0f} miles "
          f"apart (n = {len(grid_track)} hours); the 0.5° grid is ~35 miles across")
    log_decision("Open-Meteo archive (external)", "Cross-check of the storm's position",
                 "Hourly sea-level-pressure minimum on a 0.5° grid, Sept 25-27 (GMT), compared with NHC's position",
                 len(grid_wx), "Independent check that the weather data and the official track agree", len(grid_track))
grid_track.iloc[::3].head(12)

### 7.2 Store-by-store comparison

In [ ]:
PEAK = (pd.Timestamp("2024-09-24"), pd.Timestamp("2024-09-25"))
storm = joined[joined["date"].between("2024-09-25", "2024-09-28")].groupby("store_id").agg(
    storm_gust_mph=("max_gust_mph", "max"), storm_min_pressure=("min_pressure_hpa", "min"))

def surge_start(series):
    # first day after the baseline when a store's units clear its own baseline mean + 3 SD
    b = series[BASELINE_START:BASELINE_END]
    c = series[SURGE_SEARCH_START:SURGE_END]
    return c[c > b.mean() + 3 * b.std()].index.min()

store_daily = dated_sales.groupby(["date", "store_id"])["quantity"].sum().unstack()
q4 = pd.DataFrame({
    "baseline_per_day": window_rate(dated_sales, BASELINE_START, BASELINE_END, "store_id"),
    "surge_per_day": window_rate(dated_sales, SURGE_START, SURGE_END, "store_id"),
    "peak_per_day": window_rate(dated_sales, *PEAK, "store_id"),
    "surge_start": store_daily.apply(surge_start),
})
q4["lift_pct"] = 100 * (q4["surge_per_day"] / q4["baseline_per_day"] - 1)
q4["peak_multiple"] = q4["peak_per_day"] / q4["baseline_per_day"]
q4 = q4.join(stores.set_index("store_id")[["city", "region", "latitude", "longitude"]]).join(storm)
def closest_approach(lat, lon):
    d = haversine_mi(lat, lon, nhc_dense["lat"].values, nhc_dense["lon"].values)
    k = int(np.argmin(d))
    return d[k], nhc_dense.loc[k, "time_gmt"]

ca = [closest_approach(la, lo) for la, lo in zip(q4["latitude"], q4["longitude"])]
q4["miles_to_track"] = [c[0] for c in ca]                     # closest approach of the official NHC track
q4["closest_approach_edt"] = [c[1].tz_convert("America/New_York").strftime("%b %d %H:%M") for c in ca]
q4["miles_to_landfall"] = haversine_mi(q4["latitude"], q4["longitude"], STORM_LAT, STORM_LON)
q4 = q4.sort_values("lift_pct", ascending=False)
(q4[["city", "region", "baseline_per_day", "surge_per_day", "lift_pct", "peak_multiple", "miles_to_track",
     "closest_approach_edt", "miles_to_landfall", "storm_gust_mph", "storm_min_pressure"]].round(1).assign(surge_start=q4["surge_start"].dt.strftime("%b %d")).sort_values("miles_to_track"))

In [ ]:
# Does geography explain the differences? 12 stores, so these are descriptive, not significance tests.
GEO = ["miles_to_track", "miles_to_landfall", "storm_gust_mph", "storm_min_pressure"]
geo = pd.DataFrame({
    "r with lift_pct": [q4["lift_pct"].corr(q4[c]) for c in GEO],
    "Spearman with lift_pct": [q4["lift_pct"].corr(q4[c], method="spearman") for c in GEO],
    "r with peak_multiple": [q4["peak_multiple"].corr(q4[c]) for c in GEO],
}, index=GEO).round(2)
print(f"Lift range: {q4['lift_pct'].min():.0f}% ({q4['lift_pct'].idxmin()} {q4.loc[q4['lift_pct'].idxmin(), 'city']}) to "
      f"{q4['lift_pct'].max():.0f}% ({q4['lift_pct'].idxmax()} {q4.loc[q4['lift_pct'].idxmax(), 'city']}); "
      f"coefficient of variation across stores {q4['lift_pct'].std() / q4['lift_pct'].mean():.0%}")
print(f"Peak multiple range: {q4['peak_multiple'].min():.2f}x to {q4['peak_multiple'].max():.2f}x")
print("Surge start dates:", q4["surge_start"].dt.strftime("%b %d").value_counts().to_dict())
print(f"Closest store to the NHC track: {q4.loc[q4['miles_to_track'].idxmin(), 'city']} "
      f"({q4['miles_to_track'].min():.0f} mi); farthest: {q4.loc[q4['miles_to_track'].idxmax(), 'city']} "
      f"({q4['miles_to_track'].max():.0f} mi)")
near = q4.nsmallest(4, "miles_to_track"); far = q4.nlargest(4, "miles_to_track")
print(f"4 stores nearest the track: {', '.join(near['city'])} -> mean lift {near['lift_pct'].mean():.0f}%; "
      f"4 farthest: {', '.join(far['city'])} -> mean lift {far['lift_pct'].mean():.0f}%")
geo

In [ ]:
# Chart: where the stores are vs. the storm, and lift vs. distance
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.6), gridspec_kw={"width_ratios": [1.05, 1]})
a1.scatter(q4["longitude"], q4["latitude"], s=q4["lift_pct"] * 4, color=PALETTE[0], alpha=0.75, edgecolor="white",
           linewidth=1.5, label="Store (size = lift)")
OFFSETS = {"FL-105": (-12, 6, "right"), "FL-445": (-12, -10, "right"), "FL-302": (2, -20, "left"),
           "FL-287": (12, -6, "left"), "FL-239": (12, 7, "left")}     # keeps the central-Florida labels apart
for sid, r in q4.iterrows():
    dx, dy, ha = OFFSETS.get(sid, (12, -3, "left"))
    a1.annotate(f"{r['city']} +{r['lift_pct']:.0f}%", (r["longitude"], r["latitude"]), xytext=(dx, dy),
                textcoords="offset points", ha=ha, fontsize=8.5, color="#52514e")
a1.plot(nhc["lon"], nhc["lat"], color=PALETTE[1], lw=2, marker="o", ms=3.5, label="NHC best track (6-hourly)")
if len(grid_track):
    a1.scatter(grid_track["lon"], grid_track["lat"], s=14, color=MUTED, alpha=0.6, lw=0, label="Open-Meteo hourly pressure low")
a1.scatter([STORM_LON], [STORM_LAT], marker="*", s=320, color=PALETTE[1], edgecolor="white", linewidth=1, zorder=5,
           label=f"Landfall, {LANDFALL_TIME_LOCAL:%b %d %I:%M %p} EDT")
a1.set_xlim(-88, -79.2); a1.set_ylim(24.5, 32.5)
a1.set_xlabel("Longitude"); a1.set_ylabel("Latitude")
a1.set_title("Stores sized by lift, with Helene's official NHC track", fontsize=11.5)
a1.legend(loc="lower left", fontsize=8.5, markerscale=0.5)
a2.scatter(q4["miles_to_track"], q4["lift_pct"], s=50, color=PALETTE[0], edgecolor="white", linewidth=1, label="Store")
for sid, r in q4.iterrows():
    if sid in (q4["lift_pct"].idxmax(), q4["lift_pct"].idxmin(), q4["miles_to_track"].idxmin(), q4["miles_to_track"].idxmax()):
        a2.annotate(r["city"], (r["miles_to_track"], r["lift_pct"]), xytext=(6, 4), textcoords="offset points",
                    fontsize=8.5, color="#52514e")
a2.set_xlabel("Closest approach of the NHC track (miles)")
a2.set_ylabel("Lift, surge window vs. baseline (%)")
a2.set_ylim(0, q4["lift_pct"].max() * 1.15)
a2.set_xlim(0, q4["miles_to_track"].max() * 1.12)
a2.set_title(f"Lift vs. distance to the track (r = {geo.loc['miles_to_track', 'r with lift_pct']:+.2f}, "
             f"Spearman = {geo.loc['miles_to_track', 'Spearman with lift_pct']:+.2f})", fontsize=11.5)
a2.legend(loc="lower left", frameon=True)
fig.suptitle("Every store surged by about the same amount, near Helene's path or far from it",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Chart: category lift by store (heatmap, one sequential hue)
cat_store = 100 * (window_rate(dated_sales, SURGE_START, SURGE_END, ["store_id", "category"]) /
                   window_rate(dated_sales, BASELINE_START, BASELINE_END, ["store_id", "category"]) - 1)
hm = cat_store.unstack().loc[q4.index, cat_lift.index]
hm.index = [f"{q4.loc[s, 'city']} ({s})" for s in hm.index]
fig, ax = plt.subplots(figsize=(11.5, 6))
im = ax.imshow(hm.values, cmap="Blues", aspect="auto", vmin=0)
ax.set_xticks(range(hm.shape[1]), hm.columns, rotation=30, ha="right")
ax.set_yticks(range(hm.shape[0]), hm.index)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        v = hm.values[i, j]
        ax.text(j, i, f"{int(round(v))}", ha="center", va="center", fontsize=8.5,
                color="white" if v > np.nanmax(hm.values) * 0.6 else "#0b0b0b")
ax.grid(False)
ax.set_xlabel("Category (sorted by statewide lift)")
ax.set_ylabel("Store (sorted by total lift)")
n_bfast = (hm.idxmax(axis=1) == "Breakfast").sum()
ax.set_title(f"Breakfast (Pop-Tarts) has the biggest lift in {n_bfast} of {len(hm)} stores; Snacks and Baby trail almost everywhere")
cb = fig.colorbar(im, ax=ax, shrink=0.8); cb.set_label("Lift vs. baseline (%)  [color scale = legend]")
plt.tight_layout(); plt.show()

### 7.3 Can the distribution centers keep up?
Exhibit 4 of the case and `distribution_centers.csv` give each DC's normal daily shipments and its surge maximum.
We assign each store to its nearest DC, then compare the demand multiple the stores saw with the multiple each DC can surge to.
The DC file has cities but no coordinates, so we geocode the DC cities with the same Open-Meteo geocoding API we used for Tallahassee.

**Cleaning the DC file:** Miami's `surge_capacity` is the text `~750` (read as 750 and flagged approximate), and Birmingham's
`cost_per_shipment` is blank (left missing, not imputed). Both go in the decision log.

**Assumptions (back-of-envelope):**
- A DC's normal shipments supply a normal day of demand at the stores it serves.
- Shipments scale one-for-one with units sold.
- Stores can hold stock that is shipped early.

In [ ]:
dcs_raw = pd.read_csv(data_path("distribution_centers.csv"), dtype=str)
dcs = dcs_raw.copy()
approx = dcs["surge_capacity"].str.contains("~", na=False)
dcs["surge_capacity"] = pd.to_numeric(dcs["surge_capacity"].str.replace("~", "", regex=False))
dcs["surge_capacity_approx"] = approx
for c in ["avg_daily_shipments", "distance_to_orlando_mi", "cost_per_shipment"]:
    dcs[c] = pd.to_numeric(dcs[c])
DCF = "distribution_centers.csv"
log_decision(DCF, "Number stored as text", "surge_capacity '~750' (Miami) read as 750 and flagged surge_capacity_approx",
             approx.sum(), "Needed as a number for the capacity ratio; the flag keeps the approximation visible", len(dcs))
log_decision(DCF, "Missing values (cost)", "Birmingham cost_per_shipment is blank: left as NaN, not imputed",
             dcs["cost_per_shipment"].isna().sum(), "Only used to rank DCs by cost; a guessed cost could change that ranking",
             len(dcs))

STATE_NAMES = {"FL": "Florida", "GA": "Georgia", "TX": "Texas", "AL": "Alabama"}
coords = [geocode_city(c, STATE_NAMES[s]) for c, s in zip(dcs["city"], dcs["state"])]
dcs["lat"], dcs["lon"] = [c[0] for c in coords], [c[1] for c in coords]
log_decision(DCF, "Missing values (coordinates)", "No lat/lon in the file: geocoded each DC city with Open-Meteo's geocoding API",
             len(dcs), "Needed to assign each store its nearest DC; checked against the file's road miles to Orlando", len(dcs))
check_orl = haversine_mi(dcs["lat"], dcs["lon"], *stores.set_index("store_id").loc["FL-239", ["latitude", "longitude"]])
print("Sanity check, straight-line miles to Orlando vs. the file's distance_to_orlando_mi (road miles, so ours should be lower):")
print(pd.DataFrame({"geocoded_straight_line": check_orl.round(0), "file": dcs["distance_to_orlando_mi"]}, ).set_index(dcs["dc_id"]))
dcs["surge_multiple"] = dcs["surge_capacity"] / dcs["avg_daily_shipments"]

dist = pd.DataFrame({dc.dc_id: haversine_mi(q4["latitude"], q4["longitude"], dc.lat, dc.lon) for dc in dcs.itertuples()})
q4["nearest_dc"] = dist.idxmin(axis=1)
q4["miles_to_dc"] = dist.min(axis=1)
dc_view = dcs.set_index("dc_id")[["city", "avg_daily_shipments", "surge_capacity", "surge_multiple", "cost_per_shipment"]]
dc_view["stores_served"] = q4.groupby("nearest_dc").size()
dc_view["store_list"] = q4.groupby("nearest_dc")["city"].agg(", ".join)
dc_view.fillna({"stores_served": 0, "store_list": "-"}).round(2)

In [ ]:
mult = daily_units / base.mean()                                  # statewide demand as a multiple of a normal day
served = dc_view[dc_view["stores_served"] > 0]
cap_mult = np.average(served["surge_multiple"], weights=served["stores_served"])
headroom = cap_mult - 1                                           # extra "normal days" of supply a DC can add per day
build = mult[DEMAND_ONSET:SURGE_END]                              # the whole build-up: first abnormal day to landfall eve
over_cap_days = build[build > cap_mult].index
extra_needed = (build - 1).clip(lower=0).sum()
extra_window = (mult[SURGE_START:SURGE_END] - 1).clip(lower=0).sum()

def coverage(ship_start):
    # share of the build-up's extra demand the DCs can cover if surge shipping starts on ship_start
    days = max(0, (SURGE_END - ship_start).days + 1)
    return min(1, headroom * days / extra_needed)

print(f"DCs serving these stores can surge to {cap_mult:.2f}x normal (store-weighted).")
print(f"Statewide demand was above that on {len(over_cap_days)} days: "
      f"{over_cap_days.min():%b %d}-{over_cap_days.max():%b %d}, peaking at {mult.max():.2f}x.")
print(f"Extra demand before landfall ({DEMAND_ONSET:%b %d}-{SURGE_END:%b %d}) = {extra_needed:.1f} 'normal days' of volume; "
      f"{extra_window:.1f} of it ({extra_window / extra_needed:.0%}) falls in the brief's 7-day window.")
print(f"Start surge shipping when sales first rise ({DEMAND_ONSET:%b %d}) and stores hold the stock: covers ~{coverage(DEMAND_ONSET):.0%}.")
print(f"Start when the 7-day window opens ({SURGE_START:%b %d}): covers ~{coverage(SURGE_START):.0%}.")
print(f"Wait until the weather turns ({WEATHER_ALARM:%b %d}, section 3.2): covers ~{coverage(WEATHER_ALARM):.0%}.")


In [ ]:
# Chart: demand multiple vs. DC surge capacity
fig, ax = plt.subplots(figsize=(11, 4.8))
colors = [PALETTE[1] if d in over_cap_days else PALETTE[0] for d in mult.index]
ax.bar(mult.index, mult.values, color=colors, width=0.8)
ax.axhline(cap_mult, color="#0b0b0b", lw=1.2, ls="--")
ax.axhline(1, color=MUTED, lw=0.8)
ax.set_ylabel("Units sold ÷ normal (baseline) day")
ax.set_xlabel("Date (September 2024)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=PALETTE[0]), plt.Rectangle((0, 0), 1, 1, color=PALETTE[1]),
                   plt.Line2D([], [], color="#0b0b0b", ls="--"), plt.Line2D([], [], color=MUTED)],
          labels=["Demand within DC capacity", "Demand above DC capacity", f"DC surge capacity ≈ {cap_mult:.1f}× normal",
                  "Normal day (1×)"], loc="upper left")
ax.set_title(f"Demand ran above what the DCs can surge to on {len(over_cap_days)} days; the gap has to ship early")
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

### Q4 finding
*Every number here is printed by the cells above.*

**Where the storm went (official NHC best track).** Helene came up the eastern Gulf, passing about 115–120 miles west of
Tampa Bay on the afternoon and evening of Sept 26, and made landfall at **11:10 pm EDT on Sept 26 in the Big Bend near
Perry (30.0°N, 83.7°W)** as a Category 4 hurricane (120 kt, 939 mb), then moved north into Georgia. As a check, the Open-Meteo pressure field puts the storm's lowest pressure (945 hPa at 02:00 GMT on Sept 27, 29.49°N 83.97°W) about 4 miles from NHC's position at that same hour, about an hour before landfall, so the weather data and the official track agree.

**Distance to the path is not the same as distance to landfall.** Tampa and St. Petersburg are 160–167 miles from the
landfall point but only **115–122 miles from the track**, because the storm passed offshore of them. Measured to the track,
the closest stores are **Tallahassee (40 mi)**, **Gainesville (86 mi)**, **Jacksonville (114 mi)**, **St. Petersburg (115 mi)** and
**Tampa (122 mi)**. The farthest are **Miami (302 mi)** and **Pensacola (206 mi)**.

**Every store surged, and by about the same amount.**
- **Lift (Sept 19–25 vs. baseline):** from **+122% (Orlando)** to **+171% (Miami)**. The coefficient of variation across stores is only 8%.
- **Peak (Sept 24–25):** every store ran at **3.0× to 3.9×** normal.
- **Start date:** the surge began the same week everywhere: Sept 16 in 9 of 12 stores, Sept 17–18 in the other three.
- **Category mix:** Breakfast (Pop-Tarts) had the biggest lift in all 12 stores; Snacks (9 stores) or Baby (3) had the smallest.

**Geography doesn't explain the differences.**
- **Distance to the NHC track:** weak (r = 0.21), and the rank correlation is near zero (Spearman −0.08). The 4 stores
  nearest the track averaged about the same lift as the 4 farthest (about 146% vs. 149%).
- **Distance to the landfall point:** r = 0.27, Spearman −0.14.
- **The store's own storm gust:** r = −0.05. **The store's lowest pressure:** r = −0.10.

Three stores show it plainly:
- **Miami** was the farthest from the track (302 miles), had only a 54 mph gust, and had the **biggest** lift (+171%).
- **Pensacola**, at the far west of the state and 206 miles from the track, got the weakest storm winds (35 mph) and still rose **+144%**.
- **Tallahassee** was closest to the path (40 miles) and had the lowest pressure of any store (977 hPa), and rose **+160%**,
  about the same as Miami at the other end of the state.

This matches Q3: shoppers reacted to the **statewide threat**, not to how close the storm would come to their own store.

**The distribution network is the bottleneck.**
- **Who serves whom:** the nearest DCs are Jacksonville for 9 of the 12 stores, Miami for 2 and Birmingham for Pensacola.
  Atlanta (\$412.50 per shipment) and Dallas (\$687) are backup only, against \$245 from Jacksonville.
- **Capacity vs. demand:** those DCs can surge to about **2.0×** normal. Statewide demand was above that for five straight days
  (**Sept 21–25**) and peaked at **3.6×**.
- **The shortfall:** extra demand before landfall (Sept 16–25) adds up to about **11.3 normal days** of volume, 90% of it
  inside the 7-day window.
  - If DCs start surge shipping when sales first rise (**Sept 16**) and stores hold the stock, they cover about **91%** of it.
  - If they start when the 7-day window opens (**Sept 19**), about **64%**.
  - If they wait until the weather itself turns (the `WEATHER_ALARM` day, Sept 25), only about **9%**.

**What it means for leadership:**
- Allocate the surge to **every Florida store**, not just stores near the projected path.
- Start surge shipping at the **first credible Florida threat**, about ten days before landfall, and use daily sales as the trigger.
- Stage stock in stores during the early build, and pre-book Atlanta as overflow.

**Caveat:** the DC math is a ratio argument under the three assumptions above. These DCs serve far more than our 12 stores, so
it shows the *shape* of the gap, not a truck count.

## 8. Q5: Signal or artifact? The board-game spike
**The question: ** Board-game sales are rising in storm regions. The Finance VP thinks it may be an artifact:
"Monopoly and card game bundles got lumped together." The CEO has 48 hours to decide whether to ship board games.

**How we test it.** An artifact leaves fingerprints, so we look for each one. The tests run in **SQL** on the SQLite database,
queried with `sqlite3` and pandas. **Before any specific SQL we print the real table schemas** (`PRAGMA table_info`) and a few
rows: the database is not shaped like the transaction CSV (see the notes under the schema output).
We copy `inventory_and_sales.db` into memory (the file is never modified), clean its two tables with SQL views, and add our
clean `SKU_MAP` and the cleaned POS rows as tables so SQL can join across them.

| If it were an artifact... | Test |
|---|---|
| Only the POS extract would show it | Does the independent `daily_sales_summary` table show the same spike? |
| One mis-coded SKU would carry it | Lift for each Monopoly and card SKU separately |
| Bundles coded as Monopoly would change price or basket size | Transactions/day, units per transaction and price, baseline vs. surge |
| One store or register would cause it | Monopoly lift in each of the 12 stores |
| Stock wouldn't move | `inventory_levels` on-hand for board games vs. other categories |

In [ ]:
db_path = data_path("inventory_and_sales.db")
if db_path.startswith("http"):                      # not on disk: download a copy into the local data folder
    local = os.path.join(DATA_DIR, "inventory_and_sales.db")
    urllib.request.urlretrieve(db_path, local)
    db_path = local
db = sqlite3.connect(":memory:")
with sqlite3.connect(db_path) as src:
    src.backup(db)                                    # in-memory copy; the file on disk is never modified
SKU_MAP[["sku", "product_id", "product", "category"]].to_sql("sku_map", db, index=False)
(dated_sales.assign(date=dated_sales["date"].dt.strftime("%Y-%m-%d"))
 [["transaction_id", "date", "store_id", "sku", "product_id", "product_name", "category", "quantity", "unit_price"]]
 .to_sql("pos", db, index=False))

def sql(q):
    return pd.read_sql(q, db)

print("Tables:", sql("SELECT name FROM sqlite_master WHERE type = 'table'")["name"].tolist())

# Look at the actual schemas before writing specific queries: column names and grain differ from the CSV
DB_TABLES = ["daily_sales_summary", "inventory_levels"]
schemas = pd.concat({t: sql(f"PRAGMA table_info({t})")[["cid", "name", "type"]].set_index("cid") for t in DB_TABLES},
                    names=["table", "cid"])
display(schemas)
for t in DB_TABLES:
    print(f"{t}: {sql(f'SELECT COUNT(*) AS n FROM {t}')['n'][0]:,} rows; first rows:")
    display(sql(f"SELECT * FROM {t} LIMIT 3"))
print("date formats in daily_sales_summary:", sql("SELECT DISTINCT length(date) AS chars FROM daily_sales_summary")["chars"].tolist())
# Profile the database's own problems, in SQL
sql('''
SELECT 'daily_sales_summary' AS tbl, 'date stored as YYYYMMDD (not YYYY-MM-DD)' AS issue, COUNT(*) AS n_rows
  FROM daily_sales_summary WHERE date NOT LIKE '____-__-__'
UNION ALL
SELECT 'daily_sales_summary', 'negative total_revenue with positive units', COUNT(*)
  FROM daily_sales_summary WHERE total_revenue < 0 AND total_units > 0
UNION ALL
SELECT 'inventory_levels', 'SKU not in SKU_MAP (orphan)', COUNT(*)
  FROM inventory_levels i LEFT JOIN sku_map m ON i.sku = m.sku WHERE m.sku IS NULL
UNION ALL
SELECT 'inventory_levels', 'units_on_order is NULL', COUNT(*)
  FROM inventory_levels WHERE units_on_order IS NULL
''')

**What the schemas tell us (and how it shapes the SQL below).**
- `daily_sales_summary` is **store × day × category** (`total_units` is REAL, plus `total_revenue`). There is no
  `transaction_id`, `sku`, `unit_price` or timestamp, so it can be compared with the POS file only at category level, and
  only by **lift**, since its volumes are on a different scale.
- `inventory_levels` is **store × day × SKU** (`units_on_hand`, `units_on_order`, `reorder_point`), so it joins to our
  `SKU_MAP` table on `sku`.
- `date` is TEXT in two formats (`YYYY-MM-DD` and `YYYYMMDD`), so the clean views normalize it before any date filter.
- Store IDs are already `FL-###`, the same form our cleaning produces for the CSVs, so no store mapping is needed.

In [ ]:
# Clean views, in SQL
db.executescript('''
DROP VIEW IF EXISTS summary_clean;
CREATE VIEW summary_clean AS
SELECT CASE WHEN date LIKE '____-__-__' THEN date
            ELSE substr(date, 1, 4) || '-' || substr(date, 5, 2) || '-' || substr(date, 7, 2) END AS date,
       store_id, category, total_units,
       ABS(total_revenue) AS total_revenue,
       (total_revenue < 0) AS revenue_sign_fixed
FROM daily_sales_summary;

DROP VIEW IF EXISTS inventory_clean;
CREATE VIEW inventory_clean AS
SELECT i.date, i.store_id, i.sku, m.product, m.category, i.units_on_hand, i.units_on_order, i.reorder_point
FROM inventory_levels i JOIN sku_map m ON i.sku = m.sku;
''')
dup = sql("SELECT COUNT(*) AS n FROM (SELECT date, store_id, category FROM summary_clean GROUP BY 1, 2, 3 HAVING COUNT(*) > 1)")["n"][0]
grid_ok = sql("SELECT COUNT(DISTINCT date) AS days, COUNT(DISTINCT store_id) AS stores, COUNT(DISTINCT category) AS cats, COUNT(*) AS n FROM summary_clean")
assert dup == 0 and grid_ok.loc[0, "n"] == grid_ok.loc[0, "days"] * grid_ok.loc[0, "stores"] * grid_ok.loc[0, "cats"]
n_fmt = sql("SELECT COUNT(*) AS n FROM daily_sales_summary WHERE date NOT LIKE '____-__-__'")["n"][0]
n_neg = sql("SELECT SUM(revenue_sign_fixed) AS n FROM summary_clean")["n"][0]
n_orph = sql("SELECT COUNT(*) AS n FROM inventory_levels i LEFT JOIN sku_map m ON i.sku = m.sku WHERE m.sku IS NULL")["n"][0]
n_summary = sql("SELECT COUNT(*) AS n FROM summary_clean")["n"][0]
n_inv = sql("SELECT COUNT(*) AS n FROM inventory_levels")["n"][0]
n_null_order = sql("SELECT COUNT(*) AS n FROM inventory_levels WHERE units_on_order IS NULL")["n"][0]
DBS, DBI = "inventory_and_sales.db: daily_sales_summary", "inventory_and_sales.db: inventory_levels"
log_decision(DBS, "Inconsistent date formats", "Normalized YYYYMMDD dates to YYYY-MM-DD in a SQL view", n_fmt,
             "Date filters must match the POS dates; asserted no duplicates and a full 30 x 12 x 10 grid", n_summary)
log_decision(DBS, "Negative revenue", "Negative total_revenue with positive units treated as a sign error: used ABS()", n_neg,
             "Units are positive on those rows, so the sign is the error; revenue is not used in a test", n_summary)
log_decision(DBI, "Inconsistent SKUs (orphans)", "Dropped rows whose SKU has no row in SKU_MAP (left join: LEGACY-PT, "
             "MISC-001, OLD-BATT-50, UNKNOWN-99)", n_orph,
             "No product or category can be assigned, and none appear in the POS data", sql("SELECT COUNT(*) AS n FROM inventory_clean")["n"][0])
log_decision(DBI, "Missing values (units_on_order NULL)", "Left NULL, not imputed", n_null_order,
             "Q5 uses units_on_hand and reorder_point only; imputing a column we never use adds a guess with no effect", n_inv)
print("orphan SKUs dropped from inventory:", sql('''SELECT i.sku, COUNT(*) AS n FROM inventory_levels i
      LEFT JOIN sku_map m ON i.sku = m.sku WHERE m.sku IS NULL GROUP BY 1''').to_dict("records"))
grid_ok

### 8.1 Test 1: does an independent source show the same spike?
`daily_sales_summary` is a separate table from the POS file, and its volumes are on a different (larger) scale, so we
compare **lift**, not levels. The same baseline and surge windows as Q1.

In [ ]:
LIFT_SQL = '''
WITH w AS (
  SELECT category,
         SUM(CASE WHEN date BETWEEN '{b0}' AND '{b1}' THEN {u} END) / {nb}.0 AS base_per_day,
         SUM(CASE WHEN date BETWEEN '{s0}' AND '{s1}' THEN {u} END) / {ns}.0 AS surge_per_day
  FROM {t} GROUP BY category)
SELECT category, base_per_day, surge_per_day, 100.0 * (surge_per_day / base_per_day - 1) AS lift_pct FROM w
'''
win = dict(b0=f"{BASELINE_START:%Y-%m-%d}", b1=f"{BASELINE_END:%Y-%m-%d}", s0=f"{SURGE_START:%Y-%m-%d}", s1=f"{SURGE_END:%Y-%m-%d}",
           nb=(BASELINE_END - BASELINE_START).days + 1, ns=(SURGE_END - SURGE_START).days + 1)
db_lift = sql(LIFT_SQL.format(t="summary_clean", u="total_units", **win)).set_index("category")
pos_lift = sql(LIFT_SQL.format(t="pos", u="quantity", **win)).set_index("category")
daily_db = sql("SELECT date, category, SUM(total_units) AS u FROM summary_clean GROUP BY 1, 2").pivot(index="date", columns="category", values="u")
daily_pos = sql("SELECT date, category, SUM(quantity) AS u FROM pos GROUP BY 1, 2").pivot(index="date", columns="category", values="u")
xcheck = pd.DataFrame({
    "lift_pct (summary table)": db_lift["lift_pct"],
    "lift_pct (POS transactions)": pos_lift["lift_pct"],
    "daily correlation": [daily_db[c].corr(daily_pos[c]) for c in db_lift.index],
}).round(2)
xcheck["gap_pts"] = (xcheck["lift_pct (summary table)"] - xcheck["lift_pct (POS transactions)"]).round(1)
xcheck.sort_values("lift_pct (POS transactions)", ascending=False)

In [ ]:
# Chart: Entertainment, two sources, same baseline-indexed scale
idx = lambda s: s / s.loc[win["b0"]:win["b1"]].mean()
ent = pd.DataFrame({"daily_sales_summary (SQLite)": idx(daily_db["Entertainment"]),
                    "POS transactions (cleaned)": idx(daily_pos["Entertainment"])})
ent.index = pd.to_datetime(ent.index)
fig, ax = plt.subplots(figsize=(11, 4.6))
for i, c in enumerate(ent.columns):
    ax.plot(ent.index, ent[c], color=PALETTE[i], marker="o", ms=3, label=c)
ax.axhline(1, color=MUTED, lw=0.8)
ax.set_ylabel("Entertainment units ÷ baseline day")
ax.set_xlabel("Date (September 2024)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.set_ylim(0)
shade_windows(ax, arrival=STORM_ARRIVAL)
ax.set_title(f"Two independent sources show the same Entertainment spike "
             f"(daily r = {xcheck.loc['Entertainment', 'daily correlation']:.2f})")
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

### 8.2 Tests 2–4: one bad SKU, bundles, or one store?

In [ ]:
sku_test = sql(f'''
SELECT p.product_name AS product, p.sku,
       SUM(CASE WHEN date BETWEEN '{win['b0']}' AND '{win['b1']}' THEN quantity END) / {win['nb']}.0 AS base_per_day,
       SUM(CASE WHEN date BETWEEN '{win['s0']}' AND '{win['s1']}' THEN quantity END) / {win['ns']}.0 AS surge_per_day
FROM pos p
WHERE p.category = 'Entertainment'
GROUP BY 1, 2 ORDER BY 1, 2
''')
sku_test["lift_pct"] = 100 * (sku_test["surge_per_day"] / sku_test["base_per_day"] - 1)
sku_test.round(1)

In [ ]:
basket = sql(f'''
SELECT product_name AS product,
       CASE WHEN date BETWEEN '{win['b0']}' AND '{win['b1']}' THEN 'baseline' ELSE 'surge' END AS period,
       COUNT(DISTINCT transaction_id) * 1.0 /
         CASE WHEN date BETWEEN '{win['b0']}' AND '{win['b1']}' THEN {win['nb']} ELSE {win['ns']} END AS transactions_per_day,
       AVG(quantity) AS units_per_transaction,
       AVG(unit_price) AS avg_price,
       MIN(unit_price) AS min_price, MAX(unit_price) AS max_price
FROM pos
WHERE product_name IN ('Monopoly Board Game', 'Playing Cards')
  AND (date BETWEEN '{win['b0']}' AND '{win['b1']}' OR date BETWEEN '{win['s0']}' AND '{win['s1']}')
GROUP BY 1, 2 ORDER BY 1, 2
''')
basket.round(2)

In [ ]:
by_store = sql(f'''
SELECT store_id,
       SUM(CASE WHEN date BETWEEN '{win['b0']}' AND '{win['b1']}' THEN quantity END) / {win['nb']}.0 AS base_per_day,
       SUM(CASE WHEN date BETWEEN '{win['s0']}' AND '{win['s1']}' THEN quantity END) / {win['ns']}.0 AS surge_per_day
FROM pos WHERE product_name = 'Monopoly Board Game'
GROUP BY 1
''')
by_store["lift_pct"] = 100 * (by_store["surge_per_day"] / by_store["base_per_day"] - 1)
by_store = by_store.merge(stores[["store_id", "city"]]).sort_values("lift_pct")
print(f"Monopoly lift is positive in {(by_store['lift_pct'] > 0).sum()} of {len(by_store)} stores "
      f"(range {by_store['lift_pct'].min():.0f}% to {by_store['lift_pct'].max():.0f}%)")
by_store.round(1)

In [ ]:
# Chart: the bundle fingerprint test, three small panels
m = basket[basket["product"] == "Monopoly Board Game"].set_index("period")
fig, axes = plt.subplots(1, 3, figsize=(12, 4.1))
for ax, (col, label, fmt) in zip(axes, [("transactions_per_day", "Transactions per day", "{:.0f}"),
                                         ("units_per_transaction", "Units per transaction", "{:.2f}"),
                                         ("avg_price", "Average price ($)", "${:.2f}")]):
    bars = ax.bar(m.index, m[col], color=[MUTED, PALETTE[0]], width=0.6)
    for i, v in enumerate(m[col]):
        ax.text(i, v * 1.02, fmt.format(v), ha="center", va="bottom", fontsize=10)
    ax.set_title(label, fontsize=11)
    ax.set_ylabel(label)
    ax.set_xlabel("Period")
    ax.set_ylim(0, m[col].max() * 1.25)
    ax.grid(axis="x", visible=False)
axes[0].legend(bars, [f"Baseline ({BASELINE_START:%b %d}–{BASELINE_END:%d})", f"Surge ({SURGE_START:%b %d}–{SURGE_END:%d})"],
               loc="upper left", fontsize=8.5)
fig.suptitle(f"Monopoly: {m.loc['surge', 'transactions_per_day'] / m.loc['baseline', 'transactions_per_day']:.1f}x the buyers, "
             "same basket, same price. That is not a bundle mis-code",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

### 8.3 Test 5: does the inventory table confirm it?

In [ ]:
inv = sql('''
SELECT category,
       CASE WHEN date <= '{b1}' THEN '1 baseline (Sep 1-14)'
            WHEN date <  '{s0}' THEN '2 before the window (Sep 15-18)'
            WHEN date <= '{s1}' THEN '3 7-day window (Sep 19-25)'
            ELSE '4 landfall and after (Sep 26-30)' END AS phase,
       AVG(units_on_hand) AS avg_on_hand,
       AVG(units_on_hand < reorder_point) AS share_below_reorder_point,
       COUNT(*) AS records
FROM inventory_clean
GROUP BY 1, 2
'''.format(**win))
on_hand = inv.pivot(index="category", columns="phase", values="avg_on_hand")
on_hand["surge vs baseline"] = on_hand.iloc[:, 2] / on_hand.iloc[:, 0] - 1
on_hand.sort_values("surge vs baseline").round(2)

In [ ]:
# Chart: on-hand by phase for board games vs. the category that surged most vs. the one that barely moved
show = ["Entertainment", "Emergency", "Snacks"]
fig, ax = plt.subplots(figsize=(10, 4.4))
phases = on_hand.columns[:4]
x = np.arange(len(phases)); w = 0.26
for i, c in enumerate(show):
    ax.bar(x + (i - 1) * w, on_hand.loc[c, phases], width=w, color=PALETTE[i], label=c)
ax.set_xticks(x, [p[2:] for p in phases])
ax.set_ylabel("Average units on hand per store-SKU")
ax.set_xlabel("Phase")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper right")
ax.set_title("Inventory rose then fell the same way for every category, even Snacks, so it can't settle the question")
plt.tight_layout(); plt.show()

### 8.4 Scorecard and sizing the bet

In [ ]:
mono = prod_lift.loc["Monopoly Board Game"]
cards = prod_lift.loc["Playing Cards"]
mono_daily = dated_sales[dated_sales["product_name"] == "Monopoly Board Game"].groupby("date")["quantity"].sum()
mono_peak_x = mono_daily[PEAK[0]:PEAK[1]].mean() / mono_daily[BASELINE_START:BASELINE_END].mean()
after0 = LANDFALL + pd.Timedelta(days=1)
mono_after_x = mono_daily[after0:after0 + pd.Timedelta(days=1)].mean() / mono_daily[BASELINE_START:BASELINE_END].mean()
mb, ms = m.loc["baseline"], m.loc["surge"]
mono_skus = sku_test[sku_test["product"] == "Monopoly Board Game"]["lift_pct"]
ent_x = xcheck.loc["Entertainment"]
scorecard = pd.DataFrame([
    ["Independent source", "Spike only in POS", f"Summary table +{ent_x['lift_pct (summary table)']:.0f}% vs POS +{ent_x['lift_pct (POS transactions)']:.0f}%, daily r = {ent_x['daily correlation']:.2f}", "Signal"],
    ["One mis-coded SKU", "One SKU carries the lift", f"All 3 Monopoly SKUs +{mono_skus.min():.0f}% to +{mono_skus.max():.0f}%", "Signal"],
    ["Bundles coded as Monopoly", "Price or basket size shifts", f"{mb['transactions_per_day']:.0f} -> {ms['transactions_per_day']:.0f} transactions/day; {mb['units_per_transaction']:.2f} -> {ms['units_per_transaction']:.2f} units each; ${mb['avg_price']:.2f} -> ${ms['avg_price']:.2f}", "Signal"],
    ["One store or register", "Concentrated in a few stores", f"Up in {(by_store['lift_pct'] > 0).sum()}/12 stores ({by_store['lift_pct'].min():.0f}% to {by_store['lift_pct'].max():.0f}%)", "Signal"],
    ["Inventory moves", "Stock flat while 'sales' spike", "On-hand fell for every category, Snacks included", "Inconclusive"],
], columns=["test", "artifact would look like", "what we see", "verdict"])
with pd.option_context("display.max_colwidth", 120):
    display(scorecard)
print(f"Monopoly: +{mono['lift_pct']:.0f}% over the window, {mono_peak_x:.1f}x on Sept 24-25, "
      f"{mono_after_x:.1f}x on the two days after landfall | Playing cards: +{cards['lift_pct']:.0f}%")
print(f"In this 12-store POS sample: Monopoly {mono['baseline_units_per_day']:.0f} -> {mono['surge_units_per_day']:.0f} units/day "
      f"(+{mono['extra_units_per_day']:.0f}/day, about {mono['extra_units_per_day'] / 12:.1f} extra per store per day)")
print("Cheapest DCs per shipment:", dcs.dropna(subset=["cost_per_shipment"]).sort_values("cost_per_shipment")[["dc_id", "cost_per_shipment"]].head(3).to_dict("records"))

### Q5 finding

**The board-game spike is a real signal, not an artifact, but it is smaller than the headline.** Four of five tests
say "signal". The fifth can't tell either way. (Schemas were checked first: the database is store × day × category for
sales and store × day × SKU for inventory, with dates in two text formats; see the notes after the schema output.)

1. **Independent source.** The SQLite `daily_sales_summary` shows Entertainment up **+144%** over Sept 19–25, against
   **+123%** in the POS file, with a daily correlation of **0.96**. A POS coding error wouldn't show up in a separately built table.
2. **One mis-coded SKU?** No. All three Monopoly SKUs rose by similar amounts (**+133% to +173%**). Our SKU consolidation also
   never linked a card SKU to a Monopoly SKU, because cashiers never sold them under a shared name.
3. **Bundles coded as Monopoly?** No.
   - Monopoly transactions went from **28 to 75 a day (2.6×)**.
   - Units per transaction stayed flat (**1.53 → 1.49**).
   - Average price stayed flat (**\$20.01 → \$20.01**), and so did the price range.
   - A bundle rung up as Monopoly would shift the price or the basket size. Neither moved. These were more people buying the same game.
4. **Sales only in one store or register?** No. Monopoly rose in **12 of 12 stores**, from +93% to +284%.
5. **Inventory is inconclusive.** On-hand rose before the window (Sept 15–18) and then fell 14–33% below baseline during
   the window for **every** category, including Snacks (−23%), whose sales barely moved. That is a network-wide
   restock-and-draw-down pattern, so it can't single out board games; Entertainment actually drew down the least (−14%).
   We don't count it either way, and we'd want SKU-level inventory before using it.

**How big is the signal?** Monopoly ran **+157%** over the surge window (about **2.6×** normal), **4.0×** on the Sept 24–25 peak,
and still about **1.5×** for the two days after landfall. Playing cards were **+98%**. The case exhibit's "+250%" (3.5×)
sits between our window average and our peak, so the real overstatement risk is *which window you quote*, not the sales themselves.

**A flag for the data team.** The summary table and POS mostly agree within about 3–16 points. The big gaps are
**Breakfast** (+236% in the summary table vs. +410% in POS), **Electronics** (+98% vs. +151%) and, in the other direction,
**Entertainment** (+144% vs. +123%). For Pop-Tarts we trust the POS figure, because we consolidated its four SKUs ourselves;
the summary table's category roll-up can't be audited at SKU level.

**Recommendation for the CEO's 48-hour decision: ship board games, sized to the measured lift.**
- **How much:** plan for about **2.6×** normal game volume across the window and about **4×** on the two peak days. In this
  12-store sample that is about **68 extra Monopoly units a day**, roughly 5–6 per store.
- **Priority:** load them after Emergency and Pop-Tarts.
- **Where from:** ship from **Jacksonville (\$245 per shipment)** or **Miami (\$298)**, not Dallas (\$687).
- **Downside:** the Finance VP's risk is limited. Board games don't spoil, and demand held at about 1.5× after landfall, so
  leftover units sell through rather than get written off.

## 9. Our recommendations to leadership
1. **Move based on the forecast, not the weather.**
   - Buying started about 10 days before landfall and peaked 1–2 days before it, while skies were still calm (Q1).
   - Sales track gusts and pressure *two days ahead* (r = +0.71 and −0.76), not same-day weather (r ≈ 0.2) (Q3).
2. **Stock Emergency and Pop-Tarts first.** Emergency adds the most units, about 650 extra a day in the 7-day window.
   Pop-Tarts has the biggest jump, about 8× normal at the peak, which echoes the case's "sevenfold". Electronics, Alcohol,
   Health and Entertainment come next (Q1).
3. **Send it to every Florida store.** The surge was statewide and almost uniform, +122% to +171% per store, whatever the
   distance from Helene's official track or the local wind (Q4).
4. **Start surge shipping about ten days out.** Demand hit 3.6× normal, but the DCs can surge to only about 2×. Shipping from
   the first rise covers about 91% of the extra demand, starting when the 7-day window opens covers about 64%, and waiting
   for the weather to turn covers only about 9% (Q4).
5. **Stock extra board games, sized right.** The spike is real: it shows in two independent sources and in all 12 stores, with
   the same price and basket. Plan for about 2.6× normal, about 4× on the peak days, shipped from Jacksonville or Miami (Q5).
6. **Fix the data that is still causing issues in the analysis.**
   - Keep the SKU→product map in the ETL. Without it, Pop-Tarts looks like four ordinary items and water looks like the top seller (Q2).
   - Don't trust catalog costs, pack sizes or suppliers yet (Q2).
   - Reconcile the summary table's Breakfast, Electronics and Entertainment roll-ups with POS (Q5).

**What we did not prove.** One storm, one month, 12 stores. The lags and lifts describe this storm; before automating
any of this, rerun the same pipeline on other storms.

## 10. Full decision log and final self-check
Section 2.3 showed the transaction and store decisions. This is the complete, shared log across every file we touched, in the
**File · Issue · Decision · Rows affected · Reason** structure: the transactions, the store file, the product catalog, the
distribution-center file, both SQLite tables, the NHC track and the Open-Meteo cross-check.

In [ ]:
LOG_COLS = ["file", "issue", "decision", "rows_affected", "reason", "rows_after"]
with pd.option_context("display.max_colwidth", 160):
    display(pd.DataFrame(decision_log)[LOG_COLS])

In [ ]:
# Shape checks that the notebook ran end to end on clean data. These confirm completeness, not that the analysis is right.
final_check = {
    "rows: raw -> clean (only exact duplicates removed)": f"{len(raw):,} -> {len(sales):,}",
    "every store_id joins to the store file": set(sales["store_id"]) <= set(stores["store_id"]),
    "45 SKUs -> 20 products": (sales["sku"].nunique(), sales["product_id"].nunique()) == (45, 20),
    "10 clean categories": sales["category"].nunique() == 10,
    "weather rows (12 stores x 30 days)": len(weather_daily) == 360,
    "weather has a source column": "source" in weather_daily.columns,
    "sales x weather join checked (360 rows, units preserved)": len(joined) == 360,
    "Q1 surge window = the 7 days before the Sept 26 landfall": (SURGE_END - SURGE_START).days + 1 == 7
        and SURGE_END == LANDFALL - pd.Timedelta(days=1) and f"{SURGE_START:%b %d}-{SURGE_END:%b %d}",
    "Q1 ranks show relative and absolute increase": {"rank_relative", "rank_absolute"} <= set(cat_lift.columns),
    "Q2 consolidation preserves units": bool(np.isclose(*totals)),
    "Q3 best lag (gust)": f"k={best['max_gust_mph'][0]:+d}",
    "Q4 stores compared": len(q4) == 12,
    "Q4 distances use the official NHC track": len(nhc) > 0 and q4["miles_to_track"].notna().all(),
    "Q5 schemas inspected before SQL": len(schemas) > 0,
    "decision log has File/Issue/Decision/Rows/Reason": set(LOG_COLS) <= set(pd.DataFrame(decision_log).columns),
    "Q5 SQL tables loaded": sql("SELECT COUNT(*) AS n FROM sqlite_master WHERE type = 'table'")["n"][0] >= 4,
    "decisions logged": len(decision_log),
}
for k, v in final_check.items():
    print(f"  {'PASS' if v is not False else 'FAIL'}  {k}: {v}")
assert all(v is not False for v in final_check.values()), "a final check failed"
print("\nNotebook finished:", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"))

### The Charts:
| # | Section | Chart | Shows |
|---|---|---|---|
| 1 | 2.4 | Daily transactions, line | When the data starts moving |
| 2 | Q1 | Baseline vs. surge, top 5 categories, bars | Size of the surge by category |
| 3 | Q1 | Daily units, top 5 categories, lines | Shape of the surge over the month |
| 4 | Q2 | Top 10 by SKU vs. by product, bars | What fragmentation hides |
| 5 | Q2 | Lift for all 20 products, bars | Pop-Tarts leads once consolidated |
| 6 | Q3 | Sales and gusts, stacked panels | Sales peak before the wind |
| 7 | Q3 | Lag correlation profile, bars | Sales lead weather by two days |
| 8 | Q3 | Emergency units vs. future gust, scatter | The lag at store-day level |
| 9 | Q4 | Store map with the NHC track + lift vs. distance to the track | Geography doesn't explain lift |
| 10 | Q4 | Category × store lift, heatmap | Same mix everywhere |
| 11 | Q4 | Demand multiple vs. DC capacity, bars | The supply bottleneck |
| 12 | Q5 | Entertainment, two sources, lines | Independent confirmation |
| 13 | Q5 | Monopoly basket test, three panels | Not a bundle mis-code |
| 14 | Q5 | Inventory on hand by phase, bars | Why inventory is inconclusive |

## 11. Reflection


### Reflection 1: Data Quality Impact *(Esther)*
> Describe a specific data quality issue you encountered in the supplied data. How did your cleaning decision change the outcome of your analysis? What would have happened if you had not caught it?

The data quality issue with the biggest effect was SKU fragmentation. Strawberry Pop-Tarts appeared under four SKUs (PT-12, SKU#459812, POPTART-STRAW and PT_STRAWBERRY), and the file had 45 SKUs for what turned out to be 20 real products. Because the cashiers' product names were shared across SKUs, we built a mapping table from that evidence and applied it to every row with a left join, asserting that the row count, total units and total revenue did not change and that every SKU was mapped. Before consolidation, a buyer reading a SKU report for the 7-day surge window would see bottled water at the top (about 107 to 109 units a day for each of its two biggest SKUs) and the four Pop-Tarts lines as ordinary items at ranks 3, 5, 6 and 7. After consolidation, Pop-Tarts is the number one product at 377 units a day, ahead of water at 320 and batteries at 243, and its lift (+410%) is roughly double the next product's. If we had not caught it, we would have told leadership to prioritize water and treated Pop-Tarts as a minor item, which is the exact opposite of the pattern the Walmart case is famous for, and the percentage lift would have looked the same, so nothing in the numbers would have warned us.

### Reflection 2: ETL Trade-offs *(Ayman)*
> You made choices about how to standardize SKUs, handle missing timestamps, and join weather data. Pick one decision and explain: what alternative approach could you have taken, and how might it have changed your results?

The decision I would pick is how we handled the roughly 880 sale rows (3.5%) with no timestamp. We kept them and flagged them with timestamp_missing, counting them toward SKU and product totals but leaving them out of anything time-based, such as the Q1 baseline and the weather join. The alternatives were to drop them, which is simple but throws away real sales, or to impute a time, which would have meant guessing which day a sale happened in a surge where the day is the whole question. Dropping them would have understated every product total by about 3.5%, and imputing could have pushed sales into the wrong side of the surge window. We also tested our choice, because the undated sales fall across categories within two points of each category's share of all sales, so leaving them out of the time-based analysis did not change the category ranking. The cost of our approach is that the time-based figures describe 96.5% of sales rather than all of them, which we state openly in the notebook.

### Reflection 3: Pipeline Trust *(Esther and Ayman)*
> The Walmart CDO argued that the biggest risk is not the model — it is the pipeline. Based on your experience in this project, do you agree or disagree? What was the most fragile part of your data pipeline?

I agree with the CDO, and this project is a good example of why. Our models were simple, a baseline comparison and a few correlations, and almost all of our risk came from the pipeline. Timestamps came in five formats plus five spellings of missing, store ids had 60 spellings for 12 stores, categories had 37 spellings for 10, and the weather arrived in GMT while sales were in store-local time. The most fragile part was the SKU consolidation. We linked SKUs that share a normalized product name and then reviewed each group by hand, so one mistaken shared name would have merged two different products and silently changed every product-level result in Q1, Q2 and Q5, and the catalog could not be used as a check because its costs, pack sizes and suppliers contradict each other within the same product (for example, Pop-Tarts listed under Roku Inc). That is why we added assertions at every step: unmapped SKUs, conflicting categories, row counts and the 360-row weather join all fail loudly instead of passing quietly. A wrong model gives a visibly odd answer, but a wrong pipeline gives a reasonable-looking one.

### Reflection 4: Business vs. Data *(Esther and Ayman)*
> For Question 5 (board games), your team had to make a recommendation with imperfect data. How did you weigh statistical evidence against business risk? Would you have decided differently with more time or more data?

For board games, we weighed the evidence against the cost of being wrong. On the statistical side, four of five tests said the spike was real: the independent SQLite summary table showed Entertainment up +144% against +123% in the POS file over the 7-day window, all three Monopoly SKUs rose together, price and units per transaction stayed flat (about \$20.01 and 1.53 to 1.49), and Monopoly rose in all 12 stores. The fifth test, inventory, was inconclusive because stock fell for every category during the window, including ones whose sales barely moved. On the business side, the Finance VP's worry was overstock, but board games do not spoil and demand stayed at about 1.5 times normal after landfall, so leftover units should sell through. So we recommended shipping board games sized to the measured lift (about 2.6 times normal across the window and about 4 times on the two peak days) instead of the case's +250%, and loading them after Emergency and Pop-Tarts. With more time or data, we would test other storms and other years, since this is one storm over one month in 12 stores, and we would want inventory at the SKU level so that the inventory test could actually decide.

### Reflection 5: Team Collaboration *(Esther and Ayman)*
> How did your team divide the work? What would you do differently if you had another week? What was the most valuable skill each team member contributed?

We divided the work into two main parts, with Esther owning the cleaning pipeline, the decision log and the SKU mapping (and Q1, Q2 and Q5) and Ayman owning the weather API and the join, the store and geography work (Q3 and Q4), and most of the charts, with each of us reviewing the other's questions. We worked in separate notebooks and handed finished functions to the week's notebook owner, which stopped us from overwriting each other's work. If we had another week, we would get the weather API working in the first week, since two questions waited on it, and review each question as soon as it was drafted rather than at the end, and we would also try the pipeline on a second storm. Esther's most valuable skill was careful cleaning and checking, such as turning every decision into a logged step with an assertion, and Ayman's was getting the timezone-aware weather join right and turning results into clear charts.